# VF-NeRF (conditional-NF fork) — Kaggle training

Trains (or restores) a **frozen nerfacto backbone** for every scene in `SCENES`
(cell 0; default `['counter', 'kitchen', 'room']`, all Mip-NeRF 360),
then trains a **conditional Normalizing Flow** per scene that models
`P((3D point, 3D direction) | DINO feature)` against that scene's frozen NeRF,
and lets you **probe** every scene: pick points on each scene's training images
(or your own images -- run `python app/pick_points.py` locally to click them, or
hand-edit the per-scene lists in cell 6a), then generate the sampled novel views
for them in cell 6b. Cell 4 (a spiral flythrough render) is optional and
auto-skips a scene whose nerfacto was restored rather than trained.

**Evaluation reports** (each cell zips its own outputs to
`/kaggle/working/report_<name>.zip`, so a report can be downloaded the moment its
cell finishes): **3c** depth probe (what NeRF depth 0.3 / 3.0 looks like per
scene), **5b** held-out log-likelihood -- raw, depth-filtered and against a
shuffled-condition baseline, **5c** nearest-sample error (does the flow generate
the right point / direction), **6b** probe montages, **6c** DINO reconstruction
fidelity of the frozen NeRFs (cosine, with a random-patch floor).

### Before you run — REQUIRED (open the right sidebar → Settings)
1. **Accelerator → GPU T4 x2** (or **GPU P100**). Cell 0 hard-stops if this is off.
2. **Internet → On** — needs a phone-verified Kaggle account
   (Settings → Phone Verification). Cell 0 hard-stops if this is off.
3. Then **Save Version → Save & Run All (Commit)** for an unattended run — Kaggle
   allows up to 12 h and, unlike free Colab, will not reclaim the GPU mid-run.
   Everything written to `/kaggle/working/` is saved as the version's Output.

Heavy build artifacts (venv, repo, dataset) go in `/kaggle/temp/` (not persisted).
Checkpoints (+ any renders) land in `/kaggle/working/` as they are produced.

### Runtime and the 12 h cap
Per scene: ~1 h nerfacto (60k iters) if trained from scratch, plus ~45–90 min
conditional NF (30k steps, includes a ~6–12 min DINO precompute). The three
scenes in `SCENES` from scratch is **~5–7.5 h** plus ~40 min env build +
downloads. (`bonsai` is not in `SCENES` any more; add it back and its nerfacto is
free -- the repo ships a trained checkpoint for it.) If a run times out: **Save Version**, attach the partial `vf_nerf_outputs` as a Dataset
input (see "Loading an existing checkpoint" below), and **Run All** again --
any scene with a matching checkpoint is skipped, not retrained.

In [ ]:
# @title 0. Environment sanity check  (STOP if this cell raises)
import sys, platform, subprocess, os
print('system python:', sys.version)
print('platform:', platform.platform())
smi = subprocess.run(['bash','-c','nvidia-smi -L 2>/dev/null || true'], capture_output=True, text=True).stdout.strip()
print('GPU:', smi or '(none)')
if not smi:
    raise RuntimeError(
        'No GPU. Right sidebar -> Settings -> Accelerator -> GPU T4 x2 (or P100), '
        'and Internet -> On (needs a phone-verified account). Then re-run.')
net = subprocess.run(['bash','-c','curl -sI --max-time 10 https://pypi.org >/dev/null && echo ok || echo fail'], capture_output=True, text=True).stdout.strip()
print('internet:', net)
if net != 'ok':
    raise RuntimeError('No internet. Right sidebar -> Settings -> Internet -> On, then re-run.')
for d in ('/kaggle/temp', '/kaggle/working'):
    os.makedirs(d, exist_ok=True)


# Scenes to train + probe in this run (all Mip-NeRF 360, indoor/bounded -- see
# scripts/downloads/download_mipnerf360.py). Trim this list to train fewer.
SCENES = ['counter', 'kitchen', 'room']
_ALLOWED_SCENES = ('bonsai', 'counter', 'kitchen', 'room')  # == downloader's --scene choices
assert SCENES and all(s in _ALLOWED_SCENES for s in SCENES), f'bad SCENES {SCENES}'
print('scenes:', SCENES)

In [ ]:
# @title 1. Build isolated CUDA 11.7 / Python 3.10 env + tiny-cuda-nn + repo
# Mirrors the debugged Colab setup: torch==1.13.1+cu117 has no cp311 wheel and
# tiny-cuda-nn needs the CUDA 11.7 dev headers, so we build a py3.10 venv rather
# than touch Kaggle's system interpreter.
setup = r'''#!/bin/bash
set -e
export DEBIAN_FRONTEND=noninteractive

TMP=/kaggle/temp
REPO=$TMP/VF-NeRF-conditional
VENV=$TMP/venv310
mkdir -p $TMP

echo '=== add NVIDIA CUDA apt repo (for the 11.7 dev packages) ==='
UBU=$(. /etc/os-release && echo ${VERSION_ID//./})   # 2204 / 2404
if ! ls /etc/apt/sources.list.d/ | grep -qi cuda; then
  wget -qO /tmp/cuda-keyring.deb https://developer.download.nvidia.com/compute/cuda/repos/ubuntu${UBU}/x86_64/cuda-keyring_1.1-1_all.deb || \
  wget -qO /tmp/cuda-keyring.deb https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
  dpkg -i /tmp/cuda-keyring.deb
fi

echo '=== apt packages ==='
apt-get -qq update
# python3.10: default on 22.04; via deadsnakes otherwise
if ! apt-get -qq install -y python3.10 python3.10-venv python3.10-dev 2>/dev/null; then
  apt-get -qq install -y software-properties-common
  add-apt-repository -y ppa:deadsnakes/ppa
  apt-get -qq update
  apt-get -qq install -y python3.10 python3.10-venv python3.10-dev
fi
apt-get -q install -y \
    cuda-nvcc-11-7 cuda-cudart-dev-11-7 cuda-nvrtc-dev-11-7 libcublas-dev-11-7 libcufft-dev-11-7 \
    libcurand-dev-11-7 libcusolver-dev-11-7 libcusparse-dev-11-7 libnpp-dev-11-7 \
    libnvjpeg-dev-11-7 ninja-build ffmpeg

if [ ! -x /usr/local/cuda-11.7/bin/nvcc ]; then
  echo 'FATAL: /usr/local/cuda-11.7/bin/nvcc missing after apt install'
  ls -R /usr/local/cuda-11.7 2>/dev/null | head -40; apt-cache policy cuda-nvcc-11-7
  exit 1
fi

echo '=== register CUDA 11.7 lib path ==='
echo '/usr/local/cuda-11.7/lib64' > /etc/ld.so.conf.d/cuda-11-7.conf && ldconfig
# tiny-cuda-nn links against the CUDA driver lib (-lcuda). apt CUDA has only the
# stub; the real one ships with the GPU driver as libcuda.so.1 with no .so symlink.
STUB=/usr/local/cuda-11.7/lib64/stubs
REAL=$(ldconfig -p | awk '/libcuda\.so\.1/{print $NF; exit}')
[ -n "$REAL" ] && [ ! -e /usr/local/cuda-11.7/lib64/libcuda.so ] && ln -sf "$REAL" /usr/local/cuda-11.7/lib64/libcuda.so
export LIBRARY_PATH=/usr/local/cuda-11.7/lib64:$STUB${LIBRARY_PATH:+:$LIBRARY_PATH}
echo "libcuda: real=$REAL  stub=$(ls $STUB/libcuda.so 2>/dev/null)"

echo '=== venv310 ==='
[ -d $VENV ] || python3.10 -m venv $VENV
$VENV/bin/pip install -q 'setuptools<81' wheel

echo '=== torch 1.13.1+cu117 ==='
$VENV/bin/pip install -q torch==1.13.1 torchvision functorch --extra-index-url https://download.pytorch.org/whl/cu117
$VENV/bin/pip install -q ninja

echo '=== toolchain diagnostics ==='
export CUDA_HOME=/usr/local/cuda-11.7
export PATH=/usr/local/cuda-11.7/bin:$PATH
ls -d /usr/local/cuda* || true
which nvcc; nvcc --version 2>&1 | tail -2 || echo 'NO nvcc at /usr/local/cuda-11.7/bin'
gcc --version | head -1; g++ --version | head -1
# CUDA 11.7 nvcc rejects gcc>11; pin to gcc-11 if a newer default is present
if gcc -dumpversion | grep -qvE '^(9|10|11)'; then
  apt-get -qq install -y gcc-11 g++-11
  export CC=gcc-11 CXX=g++-11
  export NVCC_PREPEND_FLAGS='-ccbin g++-11'
  echo 'pinned to gcc-11'
fi

echo '=== build tiny-cuda-nn ==='
# derive the arch from the actual GPU (T4 -> 75, P100 -> 60, L4 -> 89, ...)
GPUCC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d '. ')
export TCNN_CUDA_ARCHITECTURES=${GPUCC:-75}
echo "TCNN_CUDA_ARCHITECTURES=$TCNN_CUDA_ARCHITECTURES"
# master built fine on Colab recently; fall back through the last few tags if it
# has since moved past CUDA 11.7. Override the whole list with TCNN_REFS.
TCNN_OK=
for ref in ${TCNN_REFS:-master v1.6 v1.5}; do
  [ "$ref" = master ] && spec='git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch' \
                       || spec="git+https://github.com/NVlabs/tiny-cuda-nn/@${ref}#subdirectory=bindings/torch"
  echo "--- trying tiny-cuda-nn @ $ref ---" | tee -a /kaggle/working/tcnn_build.log
  if $VENV/bin/pip install --no-build-isolation -v "$spec" >> /kaggle/working/tcnn_build.log 2>&1; then
    TCNN_OK=$ref; break
  fi
  echo "  @ $ref failed:"; grep -E 'error:|fatal error|unsupported|cannot find -l|undefined reference|ld returned' /kaggle/working/tcnn_build.log | tail -8
done
if [ -z "$TCNN_OK" ]; then
  echo '!!! tiny-cuda-nn build FAILED for every ref — full log at /kaggle/working/tcnn_build.log'
  grep -nE 'error:|fatal error|unsupported|cannot find -l|undefined reference|ld returned' /kaggle/working/tcnn_build.log | tail -40
  exit 1
fi
echo "tiny-cuda-nn OK (@ $TCNN_OK)"
$VENV/bin/python -c 'import tinycudann as t; print("tcnn import ok", t.__version__ if hasattr(t,"__version__") else "")'

echo '=== clone repo ==='
rm -rf $REPO
git clone --quiet https://github.com/itayhanoch/VF-NeRF-conditional.git $REPO

echo '=== apply known repo fixes ==='
# 1) eval call site missing the `step` arg (crashes at the first full-image eval)
sed -i 's/metrics_dict, _ = self.model.get_image_metrics_and_images(outputs, batch)/metrics_dict, _ = self.model.get_image_metrics_and_images(outputs, batch, step)/' \
    $REPO/nerfstudio/pipelines/base_pipeline.py
# 2) scripts/render.py imports get_mask_from_view_likelihood, removed with the
#    registration pipeline — dead import, never used. Strip it so ns-render works.
sed -i '/get_mask_from_view_likelihood/d' $REPO/scripts/render.py
# 3) DINOv2 hub code (facebookresearch/dinov2 @ main) now needs torch>=2.0
#    (F.scaled_dot_product_attention). Prepend a math-identical fallback to the
#    one module that torch.hub.load's DINOv2 — nerfstudio/utils/dino_features.py.
$VENV/bin/python - "$REPO/nerfstudio/utils/dino_features.py" <<'PYEOF'
import sys, pathlib
p = pathlib.Path(sys.argv[1])
s = p.read_text()
if 'sdpa-shim' not in s:
    shim = ('\n# sdpa-shim (DINOv2 main needs torch>=2.0; this stack is torch 1.13)\n'
            'import math as _m\n'
            'import torch as _t\n'
            'import torch.nn.functional as _F\n'
            'if not hasattr(_F, "scaled_dot_product_attention"):\n'
            '    def _sdpa(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):\n'
            '        sc = 1.0 / _m.sqrt(q.size(-1)) if scale is None else scale\n'
            '        a = _t.matmul(q, k.transpose(-2, -1)) * sc\n'
            '        if attn_mask is not None:\n'
            '            a = a.masked_fill(~attn_mask, float("-inf")) if attn_mask.dtype == _t.bool else a + attn_mask\n'
            '        a = a.softmax(-1)\n'
            '        if dropout_p:\n'
            '            a = _F.dropout(a, dropout_p)\n'
            '        return _t.matmul(a, v)\n'
            '    _F.scaled_dot_product_attention = _sdpa\n')
    # insert AFTER `from __future__` (must stay first statement); else after the docstring
    anchor = 'from __future__ import annotations\n'
    if anchor in s:
        s = s.replace(anchor, anchor + shim, 1)
    else:
        s = shim.lstrip() + '\n' + s
    p.write_text(s)
    print('dino_features.py: sdpa shim installed')
else:
    print('dino_features.py: sdpa shim already present')
PYEOF

echo '=== install repo + normalizing-flows ==='
cd $REPO
$VENV/bin/pip install -q --no-build-isolation -e . -e ./normalizing-flows

echo '=== SETUP COMPLETE ==='
'''
import os, subprocess
os.makedirs('/kaggle/temp', exist_ok=True)
with open('/kaggle/temp/setup.sh', 'w') as fh:
    fh.write(setup)
# pipefail so the cell sees bash's exit code, not tee's
rc = subprocess.call(
    ['bash', '-c', 'set -o pipefail; bash /kaggle/temp/setup.sh 2>&1 | tee /kaggle/working/setup.log'])
if rc != 0:
    print('--- last 60 lines of setup.log ---')
    print(subprocess.run(['tail', '-n', '60', '/kaggle/working/setup.log'], capture_output=True, text=True).stdout)
    raise RuntimeError(f'setup.sh failed (exit {rc}). See /kaggle/working/setup.log '
                       'and, for tiny-cuda-nn, /kaggle/working/tcnn_build.log')

In [ ]:
# @title 2. Config
import os, glob
TMP = '/kaggle/temp'
REPO = f'{TMP}/VF-NeRF-conditional'
VENV = f'{TMP}/venv310/bin'
WORK = '/kaggle/working'

# Per-scene paths (SCENES comes from cell 0).
SCENE_DATA = lambda s: f'{REPO}/data/mipnerf360/{s}'
COND_DIR   = lambda s: f'{WORK}/checkpoints/conditional_nf/{s}'

# nerfacto recipe mirrored from the original VF-NeRF (leosegre/VF_NeRF,
# reg_pipeline_pc.py): downscale 2, 60k iters, 1024 rays/batch, camera-opt off,
# full train split, center-method=focus, scene-scale 2. Applies to every scene.
NERFACTO_DOWNSCALE   = 2
NERFACTO_ITERS       = 60000
NERFACTO_RAYS        = 1024
NERFACTO_FORCE_RETRAIN = False   # True = ignore/delete any existing nerfacto run and retrain
COND_NF_MAX_STEPS    = 30000
COND_NF_FORCE_RETRAIN = True  # True = train even if a checkpoint is available for a scene
# --- conditional-NF architecture (a restored .pt keeps ITS OWN architecture --
# these only take effect on a scene that actually trains, hence FORCE_RETRAIN
# above when you change them) ---
COND_NF_NUM_BLOCKS = 8  # coupling (+batchnorm) blocks in the flow
COND_NF_HIDDEN_DIM = 128  # hidden width of every coupling layer's scale/translate MLPs
COND_NF_REDUCE_DIM = None  # reduce the DINO condition to this many dims via a jointly-trained MLP before conditioning the flow (None = no reduction; the flow sees the raw 384-d DINO feature)
COND_NF_REDUCE_DIVIDE_FACTOR = 8  # only used when COND_NF_REDUCE_DIM is set
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.7/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

NERF_OUTPUT_DIR = f'{WORK}/outputs'          # persisted; nerfacto nests <scene>/nerfacto/<ts>/
RENDER_DIR = f'{WORK}/renders'               # persisted
os.makedirs(RENDER_DIR, exist_ok=True)

# ---- per-scene checkpoint reuse: match by OUTPUT directory structure ----
# A scene is skipped (not retrained) if a matching checkpoint is found. Matching
# is structural -- the scene name must sit exactly where cell 5/7 themselves put
# it, mirroring the two layouts that actually occur (not a loose "scene name
# anywhere in the path" guess, which could mis-fire on e.g. a Dataset folder
# named "bathroom-photos"):
#   * nerfacto:       .../<scene>/nerfacto/<timestamp>/config.yml   (the raw
#                     nerfstudio output tree cell 5 trains into -- also what a
#                     *.tar.gz of outputs/ unpacks to)
#                     OR .../nerfacto_<scene>/config.yml             (cell 7's
#                     packaged bundle layout; also matches the repo's own
#                     checkpoints/nerfacto_<scene>/, so bonsai always resolves)
#   * conditional-NF: .../conditional_nf/<scene>/latest.pt (or cond_nf_step_*.pt)
#                     OR .../conditional_nf_<scene>/latest.pt (packaged bundle)
# Attach a Dataset (right sidebar -> Input -> Add Input) preserving one of these
# layouts intact for each scene you want to skip -- e.g. re-attach a prior run's
# vf_nerf_outputs.zip as-is (it already uses the nerfacto_<scene>/ and
# conditional_nf_<scene>/ bundle layout), or upload outputs/<scene>/ intact.
from pathlib import Path

def _nerf_scene_of(cfg_path):
    parts = Path(cfg_path).parts
    for i, seg in enumerate(parts):
        if seg == 'nerfacto' and i > 0 and parts[i - 1] in SCENES:
            return parts[i - 1]                       # .../<scene>/nerfacto/<ts>/config.yml
        if seg.startswith('nerfacto_') and seg[len('nerfacto_'):] in SCENES:
            return seg[len('nerfacto_'):]              # .../nerfacto_<scene>/config.yml
    return None

def _cond_scene_of(pt_path):
    parts = Path(pt_path).parts
    for i, seg in enumerate(parts):
        if seg == 'conditional_nf' and i + 1 < len(parts) and parts[i + 1] in SCENES:
            return parts[i + 1]                        # .../conditional_nf/<scene>/latest.pt
        if seg.startswith('conditional_nf_') and seg[len('conditional_nf_'):] in SCENES:
            return seg[len('conditional_nf_'):]         # .../conditional_nf_<scene>/latest.pt
    return None

PRETRAINED_NERF = {}            # scene -> config.yml (any attached Dataset OR the repo's own checkpoints/)
for c in (glob.glob('/kaggle/input/**/config.yml', recursive=True) +
          glob.glob(f'{REPO}/checkpoints/**/config.yml', recursive=True)):
    # ckpt may sit under nerfstudio_models/ (the usual nerfstudio layout, kept
    # intact by cell 7's packaging) or flat next to config.yml (the repo's own
    # committed checkpoints/nerfacto_bonsai/) -- accept either.
    if glob.glob(os.path.join(os.path.dirname(c), '**', '*.ckpt'), recursive=True):
        s = _nerf_scene_of(c)
        if s:
            PRETRAINED_NERF.setdefault(s, c)

INPUT_COND_PT = {}              # scene -> latest.pt / cond_nf_step_*.pt
for p in sorted(glob.glob('/kaggle/input/**/latest.pt', recursive=True) +
                glob.glob('/kaggle/input/**/cond_nf_step_*.pt', recursive=True)):
    s = _cond_scene_of(p)
    if s:
        INPUT_COND_PT.setdefault(s, p)

PRETRAINED_TARS = sorted(glob.glob('/kaggle/input/**/*.tar.gz', recursive=True) +
                         glob.glob('/kaggle/input/**/*.tgz', recursive=True))  # scene(s) inferred on extract, same structural rule

print('nerfacto restore :', PRETRAINED_NERF)
print('nerfacto tarballs:', PRETRAINED_TARS)
print('cond-NF restore  :', INPUT_COND_PT)

# ---- one downloadable archive per evaluation report ----
# GPU time is scarce, so every report cell below ends by zipping its own output
# directory into /kaggle/working/report_<name>.zip -- grab it from the Output tab
# (committed run) or the FileLink it prints (interactive run) without waiting
# for the whole notebook.
import shutil
from IPython.display import FileLink, display

def zip_report(name, src_dir):
    out = shutil.make_archive(f'{WORK}/{name}', 'zip', src_dir)
    print(f'report archive: {out} ({os.path.getsize(out) / 1e6:.1f} MB)')
    try:
        display(FileLink(os.path.relpath(out, os.getcwd())))
    except Exception:
        pass
    return out


In [ ]:
# @title 2b. Download scenes + build downscaled images
import os, subprocess
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

for s in SCENES:
    dd = SCENE_DATA(s)
    if not os.path.isdir(f'{dd}/images'):
        subprocess.run([f'{VENV}/python', 'scripts/downloads/download_mipnerf360.py',
                        '--scene', s, '--save-dir', f'{REPO}/data/mipnerf360'],
                       check=True, cwd=REPO)
    else:
        print(f'{dd}/images present, skip download')

    # nerfstudio 0.2.1 does NOT auto-generate images_<N>/; the dataparser just
    # looks for it. downscale 2 = training res (VF-NeRF recipe), 4 = spare.
    # Full res OOMs the image cache on a free-tier box.
    src = Path(dd) / 'images'
    n_src = len(list(src.glob('*')))
    for factor in sorted({NERFACTO_DOWNSCALE, 4}):
        dst = Path(dd) / f'images_{factor}'
        if dst.is_dir() and len(list(dst.glob('*'))) == n_src:
            print(f'  {s}/images_{factor} present, skip')
            continue
        dst.mkdir(exist_ok=True)
        def _f(p, factor=factor, dst=dst):
            im = Image.open(p); w, h = im.size
            im.resize((w // factor, h // factor), Image.LANCZOS).save(dst / p.name)
        with ThreadPoolExecutor(max_workers=8) as ex:
            list(ex.map(_f, sorted(src.glob('*'))))
        print(f'  {s}/images_{factor}: {len(list(dst.glob("*")))}')

In [ ]:
# @title 3. Train (or restore) the frozen nerfacto backbone -- per scene
import glob, subprocess, os, tarfile, re, shutil

def newest_config(s):
    # only a config whose run dir actually holds a loadable checkpoint counts --
    # this skips a half-trained run and, importantly, a stale malformed restore
    # dir (e.g. a bare `nerfacto_<scene>/` with no nerfstudio_models/), so the
    # restore branch below can rebuild it correctly.
    for pat in (f'{NERF_OUTPUT_DIR}/{s}/nerfacto/*/config.yml',
                f'{WORK}/**/{s}/nerfacto/*/config.yml'):
        c = sorted(p for p in glob.glob(pat, recursive=True)
                   if glob.glob(os.path.join(os.path.dirname(p), 'nerfstudio_models', '*.ckpt')))
        if c:
            return c[-1]
    return None

def repoint_output_dir(cfg_path, target=f'{WORK}/outputs'):
    # configs from another machine bake in an absolute output_dir; repoint it so
    # nerfstudio resolves <output_dir>/<exp>/nerfacto/<ts>/nerfstudio_models here.
    # nerfstudio serialises a Path as a multi-line !!python/object/apply:pathlib
    # .PosixPath + block-sequence-of-parts; a from-scratch config uses a plain
    # scalar. Pick the branch by which FORM is present -- not by whether re.sub
    # changed anything (a config already pointing here yields an identical sub).
    s = open(cfg_path).read()
    pathlib_pat = r'output_dir:(?: &\S+)? !!python/object/apply:pathlib\.PosixPath\n(?:- .*\n)+'
    block = ('output_dir: !!python/object/apply:pathlib.PosixPath\n'
             + ''.join(f'- {p}\n' for p in ['/'] + target.strip('/').split('/')))
    if re.search(pathlib_pat, s):
        s = re.sub(pathlib_pat, block, s, count=1)
    elif re.search(r'^output_dir: .+$', s, flags=re.M):
        s = re.sub(r'^output_dir: .+$', f'output_dir: {target}', s, count=1, flags=re.M)
    else:
        raise RuntimeError(f'no output_dir key found in {cfg_path}')
    open(cfg_path, 'w').write(s)

# Extract any attached tarballs once, up front -- each may contain one or more
# scenes' outputs/<scene>/nerfacto/<ts>/ trees; newest_config() below then finds
# whichever scenes they cover.
for tar in PRETRAINED_TARS:
    print('Extracting nerfacto tarball', tar)
    with tarfile.open(tar) as t:
        t.extractall(WORK)
for cfg in glob.glob(f'{WORK}/**/nerfacto/*/config.yml', recursive=True):
    repoint_output_dir(cfg)

NERF_CONFIGS, NERF_RESTORED = {}, {}
for s in SCENES:
    has_restore_src = (newest_config(s) is not None) or (s in PRETRAINED_NERF)
    if NERFACTO_FORCE_RETRAIN and not has_restore_src:
        shutil.rmtree(f'{NERF_OUTPUT_DIR}/{s}', ignore_errors=True)
        print(s, ': NERFACTO_FORCE_RETRAIN, cleared', f'{NERF_OUTPUT_DIR}/{s}')

    cfg = newest_config(s)

    if cfg is None and s in PRETRAINED_NERF:
        src_run = os.path.dirname(PRETRAINED_NERF[s])
        # nerfstudio's eval_setup recomputes the checkpoint dir from the config's
        # own output_dir/experiment_name/method_name/timestamp/relative_model_dir
        # -> the run directory MUST be named after the config's `timestamp`, and
        # the ckpt MUST sit under nerfstudio_models/. Neither holds for a packaged
        # `nerfacto_<scene>/` bundle or the repo's flat committed checkpoint, so
        # rebuild the canonical layout rather than copying the folder as-is.
        cfg_txt = open(f'{src_run}/config.yml').read()
        m = re.search(r'^timestamp:\s*(\S+)\s*$', cfg_txt, flags=re.M)
        ts = m.group(1) if m else 'restored'
        dst_run = f'{NERF_OUTPUT_DIR}/{s}/nerfacto/{ts}'
        print(s, ': restoring nerfacto from', src_run, '->', dst_run)
        os.makedirs(f'{dst_run}/nerfstudio_models', exist_ok=True)
        shutil.copy(f'{src_run}/config.yml', f'{dst_run}/config.yml')
        dpt = f'{src_run}/dataparser_transforms.json'
        if os.path.isfile(dpt):
            shutil.copy(dpt, f'{dst_run}/dataparser_transforms.json')
        ckpts = glob.glob(f'{src_run}/**/*.ckpt', recursive=True)
        assert ckpts, f'no *.ckpt found under {src_run}'
        for ck in ckpts:
            shutil.copy(ck, f'{dst_run}/nerfstudio_models/{os.path.basename(ck)}')
        repoint_output_dir(f'{dst_run}/config.yml')
        cfg = newest_config(s)

    NERF_RESTORED[s] = cfg is not None

    if cfg:
        print(s, ': using frozen nerfacto checkpoint', cfg)
    else:
        # nerfacto args mirrored from original VF-NeRF reg_pipeline_pc.py (minus the
        # stripped --nf-first-iter / --predict-view-likelihood which belonged to the
        # in-nerfacto NF that this fork replaced with a standalone trainer).
        cmd = [f'{VENV}/ns-train', 'nerfacto', '--data', SCENE_DATA(s),
               '--output-dir', NERF_OUTPUT_DIR, '--vis', 'tensorboard',
               '--viewer.quit-on-train-completion', 'True',
               '--max-num-iterations', str(NERFACTO_ITERS),
               '--pipeline.datamanager.train-num-rays-per-batch', str(NERFACTO_RAYS),
               '--pipeline.datamanager.camera-optimizer.mode', 'off',
               'nerfstudio-data',
               '--downscale-factor', str(NERFACTO_DOWNSCALE),
               '--center-method', 'focus',
               '--orientation-method', 'up',
               '--auto-scale-poses', 'True',
               '--scene-scale', '2']
        # NOTE: original VF-NeRF also passed --train-split-fraction 1.0, but it had
        # explicit train/eval transform files. With a single transforms.json, 1.0
        # leaves 0 eval cameras -> nerfacto's periodic eval AND ns-render (cell 4)
        # both crash. Keeping the 0.9 default (~90% train imgs) instead.
        print(s, ':', ' '.join(cmd)); subprocess.run(cmd, check=True, cwd=REPO)
        cfg = newest_config(s)

    assert cfg, f'no nerfacto config produced for {s}'
    # nerfstudio loads <dir(cfg)>/nerfstudio_models/step-*.ckpt -- and the dir
    # name must equal the config's timestamp (eval_setup rebuilds the path from
    # the config, it does not trust cfg's location). Verify both here.
    run_dir = os.path.dirname(cfg)
    ck = glob.glob(f'{run_dir}/nerfstudio_models/*.ckpt')
    ts_ok = re.search(r'^timestamp:\s*(\S+)\s*$', open(cfg).read(), flags=re.M)
    ts_ok = ts_ok and ts_ok.group(1) == os.path.basename(run_dir)
    assert ck and ts_ok, (
        f'{s}: bad nerfacto layout -- ckpt={bool(ck)}, dirname matches timestamp={bool(ts_ok)} '
        f'({run_dir})')
    print(s, ': NERF_CONFIG =', cfg)
    print(s, ': checkpoint  =', ck[-1])
    NERF_CONFIGS[s] = cfg

In [ ]:
# @title 3b. Evaluate frozen NeRF quality on the held-out set (per scene)
# Standard novel-view metrics (PSNR / SSIM / LPIPS) over nerfstudio's held-out
# split via `ns-eval` -- a sanity check on the frozen backbone before trusting
# anything the conditional NF builds on top of it. Restored checkpoints work too
# (eval_setup rebuilds paths from the config). LPIPS weights download on first use.
import subprocess, os, json
EVAL_DIR = f'{WORK}/eval'
os.makedirs(EVAL_DIR, exist_ok=True)
NERF_METRICS = {}
for s in SCENES:
    out = f'{EVAL_DIR}/{s}_nerf_metrics.json'
    cmd = [f'{VENV}/ns-eval', '--load-config', NERF_CONFIGS[s], '--output-path', out]
    print(f'--- {s} ---'); print(' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=REPO)
    NERF_METRICS[s] = json.load(open(out))['results']
    print(s, ':', {k: round(v, 4) for k, v in NERF_METRICS[s].items()})
print(json.dumps(NERF_METRICS, indent=2))


In [ ]:
# @title 3c. Depth probe -- what do NeRF depths 0.3 and 3.0 look like in each scene?
# The conditional-NF evals below filter samples by their frozen-NeRF depth
# (DEPTH_RANGE, normalized scene units), because the catastrophic log-probs are
# rays whose depth is nonsense (~0.16, or ~200 for the unbounded background).
# This cell makes the bounds concrete per scene: it renders one training frame,
# reads the NeRF depth at its centre pixel, then slides the same camera along the
# centre ray so that point sits at each DEPTH_RANGE bound and renders again. The
# PNG shows the three views (centre depth in the titles) plus the original view's
# depth map with contour lines at the bounds. These rooms are indoors, so the 3.0
# camera is usually outside the room -- that panel is expected to look wrong.
import subprocess, os, json, glob
from IPython.display import Image as _Img, display

DEPTH_RANGE = (0.3, 3.0)      # normalized scene units; reused by cells 5b / 5c
DEPTH_PROBE_DIR = f'{WORK}/depth_probe'
os.makedirs(DEPTH_PROBE_DIR, exist_ok=True)
subprocess.run([f'{VENV}/pip', 'install', '-q', 'matplotlib'], check=True)

DEPTH_PROBE = {}
for s in SCENES:
    png = f'{DEPTH_PROBE_DIR}/{s}_depth_probe.png'
    cmd = [f'{VENV}/python', '-u', 'scripts/depth_probe.py',
           '--nerf-config', NERF_CONFIGS[s], '--scene-dir', SCENE_DATA(s),
           '--output-png', png, '--depths', *map(str, DEPTH_RANGE)]
    print(f'--- {s} ---'); print(' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=REPO, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    DEPTH_PROBE[s] = json.load(open(png.replace('.png', '.json')))
    display(_Img(filename=png))

print(f"{'scene':<10}{'frame':<16}{'centre depth':>13}{'p5':>8}{'p50':>8}{'p95':>8}"
      + ''.join(f"{'@' + str(t):>12}" for t in DEPTH_RANGE))
for s, d in DEPTH_PROBE.items():
    q = d['view_depth_percentiles']
    print(f"{s:<10}{d['frame']:<16}{d['center_depth']:>13.3f}{q['p5']:>8.3f}{q['p50']:>8.3f}{q['p95']:>8.3f}"
          + ''.join(f"{t['rendered_center_depth']:>12.3f}" for t in d['targets']))
print('(p5/p50/p95 = depth percentiles over the probed view; @t = centre depth read back after moving the camera to t)')
zip_report('report_depth_probe', DEPTH_PROBE_DIR)

In [ ]:
# @title 4. (optional) Render a spiral flythrough of each frozen NeRF
# Auto-skipped per scene when nerfacto was restored from a checkpoint -- the
# flythrough is only useful to eyeball a fresh training run. Flip FORCE_RENDER
# to render every scene anyway.
import subprocess, glob
FORCE_RENDER = False

for s in SCENES:
    if NERF_RESTORED[s] and not FORCE_RENDER:
        print(s, ': nerfacto restored from a checkpoint -> skipping flythrough render.')
        continue
    out = f'{RENDER_DIR}/{s}_spiral.mp4'
    cmd = [f'{VENV}/ns-render', '--load-config', NERF_CONFIGS[s], '--traj', 'spiral',
           '--rendered-output-names', 'rgb', 'depth', '--output-path', out, '--seconds', '6']
    print(s, ':', ' '.join(cmd))
    r = subprocess.run(cmd, cwd=REPO)
    print(s, ': spiral render exit', r.returncode)

print('set FORCE_RENDER = True in this cell to render restored scenes anyway.')
print('renders present:', glob.glob(f'{RENDER_DIR}/*.mp4'))

In [ ]:
# @title 5. Train (or load) the conditional Normalizing Flow per scene -> /kaggle/working/checkpoints/
import subprocess, os, shutil

COND_LATEST = {}
for s in SCENES:
    os.makedirs(COND_DIR(s), exist_ok=True)
    COND_LATEST[s] = f'{COND_DIR(s)}/latest.pt'

    if INPUT_COND_PT.get(s) and not COND_NF_FORCE_RETRAIN:
        # train_conditional_nf.py has no resume flag, so a provided .pt is used as-is
        # (for the explorer / packaging); it is not continued. NB: a .pt trained
        # before the native-res DINO change is a different feature distribution --
        # set COND_NF_FORCE_RETRAIN = True in cell 2 to retrain instead.
        src = INPUT_COND_PT[s]
        print(s, ': using provided conditional-NF checkpoint', src)
        if os.path.abspath(src) != os.path.abspath(COND_LATEST[s]):
            shutil.copy(src, COND_LATEST[s])
        continue

    cmd = [f'{VENV}/python', '-u', 'scripts/train_conditional_nf.py',
           '--nerf-config', NERF_CONFIGS[s],
           '--scene-dir', SCENE_DATA(s),
           '--checkpoint-dir', COND_DIR(s),
           '--max-steps', str(COND_NF_MAX_STEPS),
           '--num-blocks', str(COND_NF_NUM_BLOCKS),
           '--hidden-dim', str(COND_NF_HIDDEN_DIM),
           '--batch-size', '4096']
    if COND_NF_REDUCE_DIM is not None:
        cmd += ['--reduce-dim', str(COND_NF_REDUCE_DIM), '--reduce-divide-factor', str(COND_NF_REDUCE_DIVIDE_FACTOR)]
    print(s, ':', ' '.join(cmd))
    # -u + PYTHONUNBUFFERED so the 'step N/30000 | loss ...' lines stream live
    # rather than sitting in a block buffer for minutes. The first ~6-12 min is
    # a silent DINO precompute -- every training image at native resolution into
    # one memory-mapped dino_grids_*.npy (~1.7 GB) under SCENE_DATA(s)/dino_cache/
    # (a per-image progress line prints as it goes; re-runs reuse the file).
    # NB: the NF loss is a NEGATIVE log-likelihood of a CONTINUOUS density -- it
    # legitimately goes negative (e.g. ~ -4) as the flow sharpens. Watch that
    # the trend is downward and finite, not the sign.
    subprocess.run(cmd, check=True, cwd=REPO,
                   env={**os.environ, 'PYTHONUNBUFFERED': '1'})

for s in SCENES:
    print(f'--- {s} ---')
    !ls -la {COND_DIR(s)}

In [ ]:
# @title 5b. Held-out log-likelihood: raw, depth-filtered, and shuffled-condition baseline (per scene)
# Scores the flow's own training objective -- log_prob of (frozen-NeRF surface
# point, ray direction) conditioned on that pixel's DINO feature -- on the TRAIN
# frames it was fitted on and on nerfstudio's held-out frames. Three views of the
# same samples:
#   raw       every finite sample (mean / var here are dominated by a handful of
#             bad-depth rays -- see the tail lines and cell 3c);
#   filtered  only samples whose NeRF depth is inside DEPTH_RANGE (cell 3c);
#   shuffled  the SAME targets scored under the DINO feature of another random
#             pixel of the batch. If this sits close to the true log-prob, the
#             flow has learnt the scene's surface layout and ignores the feature.
# 100 batches x 4096 = 409,600 pixels per split; the one-time DINO cache per split
# dominates the runtime (the train cache is reused from training).
import subprocess, os, json
DEPTH_RANGE = globals().get('DEPTH_RANGE', (0.3, 3.0))
LL_DIR = f'{WORK}/eval/cond_nf_likelihood'
os.makedirs(LL_DIR, exist_ok=True)
NUM_BATCHES, BATCH_SIZE, WORST_K = 100, 4096, 20

COND_NF_LL = {}
for s in SCENES:
    out = f'{LL_DIR}/{s}_cond_nf_ll.json'
    cmd = [f'{VENV}/python', '-u', 'scripts/eval_cond_nf_likelihood.py',
           '--nerf-config', NERF_CONFIGS[s], '--scene-dir', SCENE_DATA(s),
           '--cond-nf-checkpoint', COND_LATEST[s], '--output-path', out,
           '--num-batches', str(NUM_BATCHES), '--batch-size', str(BATCH_SIZE),
           '--worst-k', str(WORST_K), '--depth-range', *map(str, DEPTH_RANGE), '--shuffled']
    print(f'--- {s} ---'); print(' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=REPO, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    COND_NF_LL[s] = json.load(open(out))

def _ll_table(results, depth_range):
    hdr = (f"{'scene':<9}{'split':<6}{'| raw mean':>11}{'var':>10}{'median':>9}{'n':>8}"
           f"{'| filt mean':>12}{'var':>10}{'median':>9}{'kept%':>7}"
           f"{'| shuf mean':>12}{'var':>10}{'| shuf-filt':>12}{'var':>10}"
           f"{'| gap raw':>10}{'filt':>8}")
    print(f'depth filter = {list(depth_range)};  gap = true mean - shuffled mean (raw / filtered)')
    print(hdr); print('-' * len(hdr))
    for s, d in results.items():
        for split, r in d['splits'].items():
            f, sh = r.get('depth_filtered', {}), r.get('shuffled', {})
            fsh = f.get('shuffled', {})
            print(f"{s:<9}{split:<6}{r['mean_log_prob']:>11.3f}{r['var_log_prob']:>10.1f}"
                  f"{r['median_log_prob']:>9.3f}{r['n_samples']:>8d}"
                  f"{f.get('mean_log_prob', float('nan')):>12.3f}{f.get('var_log_prob', float('nan')):>10.1f}"
                  f"{f.get('median_log_prob', float('nan')):>9.3f}{100 * f.get('frac_kept', float('nan')):>7.1f}"
                  f"{sh.get('mean', float('nan')):>12.3f}{sh.get('var', float('nan')):>10.1f}"
                  f"{fsh.get('mean', float('nan')):>12.3f}{fsh.get('var', float('nan')):>10.1f}"
                  f"{sh.get('mean_true_minus_shuffled', float('nan')):>10.3f}"
                  f"{fsh.get('mean_true_minus_shuffled', float('nan')):>8.3f}")
        gaps = [(k, d[k]) for k in ('train_minus_test', 'median_train_minus_test',
                                    'filtered_train_minus_test', 'filtered_median_train_minus_test') if k in d]
        print(f"{'':<9}{'gap':<6}train-test: " + '   '.join(f'{k.replace("_train_minus_test", "") or "mean"} {v:+.3f}' for k, v in gaps))
    print('\ntail accounting -- how much of the RAW variance a handful of samples own:')
    for s, d in results.items():
        for split, r in d['splits'].items():
            t = r['tail_contribution']['lowest_0.001pct']
            print(f"  {s:<9}{split:<6} lowest {t['n']:>3} (<= {t['threshold']:>10.1f}): "
                  f"{100 * t['variance_share']:>5.1f}% of var, {t['mean_shift']:+.3f} of mean")

_ll_table(COND_NF_LL, DEPTH_RANGE)
zip_report('report_cond_nf_likelihood', LL_DIR)

In [ ]:
# @title 5c. Nearest-sample error: does the flow GENERATE the right point / direction? (per scene)
# The flow run the other way round: for NUM_PIXELS random (image, sub-pixel)
# draws per split, take the pixel's DINO feature, sample NUM_SAMPLES (point,
# direction) candidates from the flow, and measure against the true target (the
# frozen-NeRF surface point along the pixel's ray, and the ray direction):
#   min  the closest of the K samples -- distance (scene units) and angle (deg),
#        each minimised separately: "did ANY sample land near the truth?"
#   top  the single sample the flow ranks highest by log_prob -- what the
#        explorer renders first: "is its best guess right?"
# A pixel is a HIT when the variant's distance is under HIT_FRAC x backoff
# (backoff = median camera distance from the scene centre, the explorer's
# constant). Every statistic is repeated over only the pixels whose NeRF depth
# is inside DEPTH_RANGE (cell 3c). 40k x 100 = 4M flow samples: ~1-2 min per split.
import subprocess, os, json
DEPTH_RANGE = globals().get('DEPTH_RANGE', (0.3, 3.0))
NS_DIR = f'{WORK}/eval/cond_nf_nearest_sample'
os.makedirs(NS_DIR, exist_ok=True)
NUM_PIXELS, NUM_SAMPLES_NS, HIT_FRAC = 40000, 100, 0.05

COND_NF_NS = {}
for s in SCENES:
    out = f'{NS_DIR}/{s}_cond_nf_nearest.json'
    cmd = [f'{VENV}/python', '-u', 'scripts/eval_cond_nf_nearest_sample.py',
           '--nerf-config', NERF_CONFIGS[s], '--scene-dir', SCENE_DATA(s),
           '--cond-nf-checkpoint', COND_LATEST[s], '--output-path', out,
           '--num-pixels', str(NUM_PIXELS), '--num-samples', str(NUM_SAMPLES_NS),
           '--hit-frac', str(HIT_FRAC), '--depth-range', *map(str, DEPTH_RANGE)]
    print(f'--- {s} ---'); print(' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=REPO, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    COND_NF_NS[s] = json.load(open(out))

def _ns_table(results, depth_range):
    hdr = (f"{'scene':<9}{'split':<6}{'var':<5}{'| dist mean':>12}{'var':>10}{'median':>9}"
           f"{'| hits':>8}{'%':>7}{'| angle mean':>13}{'var':>10}{'median':>9}"
           f"{'| filt dist mean':>17}{'var':>10}{'median':>9}{'hits%':>7}{'| angle mean':>13}{'median':>9}{'kept%':>7}")
    print(f"'hits' = pixels whose distance < HIT_FRAC x backoff; angle in degrees; "
          f"'filt' = pixels with NeRF depth in {list(depth_range)}")
    print(hdr); print('-' * len(hdr))
    nan = float('nan')
    for s, d in results.items():
        for split, r in d['splits'].items():
            f = r.get('depth_filtered', {})
            for v in ('min', 'top'):
                st = r['variants'][v]
                ft = f.get('variants', {}).get(v, {})
                print(f"{s:<9}{split:<6}{v:<5}{st['dist']['mean']:>12.4f}{st['dist']['var']:>10.4f}"
                      f"{st['dist']['median']:>9.4f}{st['n_hit']:>8d}{100 * st['frac_hit']:>7.1f}"
                      f"{st['angle_deg']['mean']:>13.2f}{st['angle_deg']['var']:>10.1f}{st['angle_deg']['median']:>9.2f}"
                      f"{ft.get('dist', {}).get('mean', nan):>17.4f}{ft.get('dist', {}).get('var', nan):>10.4f}"
                      f"{ft.get('dist', {}).get('median', nan):>9.4f}{100 * ft.get('frac_hit', nan):>7.1f}"
                      f"{ft.get('angle_deg', {}).get('mean', nan):>13.2f}{ft.get('angle_deg', {}).get('median', nan):>9.2f}"
                      f"{100 * f.get('frac_kept', nan):>7.1f}")
        r0 = next(iter(d['splits'].values()))
        print(f"{'':<9}backoff {r0['backoff']:.4f}  hit threshold {r0['hit_threshold']:.4f}  "
              f"({d['num_pixels']} pixels x {d['num_samples']} samples per split)")

_ns_table(COND_NF_NS, DEPTH_RANGE)
zip_report('report_cond_nf_nearest_sample', NS_DIR)

In [ ]:
# @title 6a. Pick points to probe (per scene)
# BEST: run  python app/pick_points.py --scene <name>  on your own machine -- click
# the objects you want to probe and paste the COORDS block it prints in below.
# (Kaggle's JupyterLab has no ipympl widget frontend, so there is no in-notebook
# click capture.)
# COORDS is a dict: one list of picks per scene in SCENES. Each pick is
# [ref, x, y], [ref, x, y, tag] or [ref, x, y, tag, backoff], x/y in that
# image's own pixels:
#   * ref = 'DSCF####.JPG'  -> a training frame of that scene (matched by filename
#     in images_2/)
#   * ref = 'myphoto.jpg'   -> any other image; attach a Kaggle Dataset containing
#     it (right sidebar -> Input -> Add Input). Only its DINO feature at the pixel
#     is used -- the rendered views are still that scene's renders.
#   * ref = int             -> old form: index into sorted(images_2/*)
# The tag is the picker's 'TRAIN' / 'TEST' / 'EXTERNAL' label for where the image
# came from -- TRAIN/TEST being the split THIS notebook's dataparser makes. It is
# documentation: nothing acts on it, but a TEST or EXTERNAL probe has no camera in
# the train split, so it always falls back to the constant backoff below.
# An optional field after it sets the render-camera backoff (how far behind the
# sampled point the novel-view camera sits, in scene units). A bare number in the
# 4th slot is still read as that backoff, for blocks picked before tags existed:
#   * omitted or <= 0  -> auto: the frozen NeRF's depth along that pixel's real ray
#     for a training frame, else the scene-wide constant (printed by cell 6b).
#   * > 0              -> use that literal backoff for this probe (also the only way
#     to control standoff for an external image).
# Referencing frames by FILENAME (not position) keeps this aligned with the local
# picker (app/pick_points.py emits the 4 tagged fields; add the backoff by hand).
# A TEST frame is held out, so it has no camera in the train split and cell 6b
# treats it like an external image: there is no real ray to read the frozen NeRF's
# depth along. This cell
# shows each scene's GRID_VIEW full-size with a pixel grid, then redraws every
# pick as a numbered circle. It writes /kaggle/working/coords.json for cell 6b.
# A scene absent from COORDS (or mapped to an empty list) is skipped in 6b.
import glob, json, math, os
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from PIL import Image

COORDS = {
    'counter': [
        ["DSCF5857.JPG", 466, 92, "TRAIN"],                 # counter frame 0
        ["DSCF5857.JPG", 1104, 112, "TRAIN"],               # counter frame 0
        ["DSCF5857.JPG", 372, 490, "TRAIN"],                # counter frame 0
        ["DSCF5863.JPG", 548, 595, "TEST"],                 # counter frame 6
        ["DSCF5863.JPG", 1142, 458, "TEST"],                # counter frame 6
        ["DSCF5863.JPG", 1461, 312, "TEST"],                # counter frame 6
        ["DSCF5863.JPG", 1273, 794, "TEST"],                # counter frame 6
        ["DSCF5863.JPG", 831, 54, "TEST"],                  # counter frame 6
        ["DSCF5867.JPG", 773, 144, "TRAIN"],                # counter frame 10
        ["DSCF5867.JPG", 1412, 437, "TRAIN"],               # counter frame 10
        ["DSCF5867.JPG", 1259, 832, "TRAIN"],               # counter frame 10
        ["DSCF5867.JPG", 380, 586, "TRAIN"],                # counter frame 10
        ["DSCF5886.JPG", 689, 819, "TEST"],                 # counter frame 29
        ["DSCF5886.JPG", 578, 124, "TEST"],                 # counter frame 29
        ["DSCF5887.JPG", 704, 823, "TRAIN"],                # counter frame 30
        ["DSCF5887.JPG", 595, 193, "TRAIN"],                # counter frame 30
        ["DSCF5927.JPG", 484, 155, "TRAIN"],                # counter frame 70
        ["DSCF5927.JPG", 831, 354, "TRAIN"],                # counter frame 70
        ["DSCF5937.JPG", 126, 533, "TRAIN"],                # counter frame 80
        ["DSCF5936.JPG", 118, 575, "TEST"],                 # counter frame 79
        ["DSCF5947.JPG", 288, 264, "TRAIN"],                # counter frame 90
        ["DSCF5984.JPG", 486, 287, "TRAIN"],                # counter frame 127
        ["DSCF5984.JPG", 180, 375, "TRAIN"],                # counter frame 127
        ["DSCF5984.JPG", 894, 317, "TRAIN"],                # counter frame 127
        ["DSCF6006.JPG", 429, 217, "TEST"],                 # counter frame 149
        ["DSCF6006.JPG", 499, 317, "TEST"],                 # counter frame 149
        ["DSCF6026.JPG", 897, 486, "TEST"],                 # counter frame 169
        ["DSCF6027.JPG", 923, 581, "TRAIN"],                # counter frame 170
        ["DSCF6047.JPG", 449, 305, "TRAIN"],                # counter frame 190
        ["DSCF6046.JPG", 365, 389, "TEST"],                 # counter frame 189
        ["apples.jpeg", 123, 366, "EXTERNAL"],              # attach a Kaggle Dataset containing 'apples.jpeg'
        ["cardboard box.jpg", 132, 60, "EXTERNAL"],         # attach a Kaggle Dataset containing 'cardboard box.jpg'
        ["cardboard box.jpg", 131, 97, "EXTERNAL"],         # attach a Kaggle Dataset containing 'cardboard box.jpg'
        ["eggs carton.jpg", 178, 111, "EXTERNAL"],          # attach a Kaggle Dataset containing 'eggs carton.jpg'
        ["green-pringles.png", 72, 72, "EXTERNAL"],         # attach a Kaggle Dataset containing 'green-pringles.png'
        ["metal bowe.jpg", 89, 99, "EXTERNAL"],             # attach a Kaggle Dataset containing 'metal bowe.jpg'
        ["metal bowe.jpg", 75, 62, "EXTERNAL"],             # attach a Kaggle Dataset containing 'metal bowe.jpg'
        ["silver refrigirator.jpg", 176, 256, "EXTERNAL"],  # attach a Kaggle Dataset containing 'silver refrigirator.jpg'
    ],
    'kitchen': [
        ["DSCF0656.JPG", 1358, 303, "TRAIN"],                  # kitchen frame 0
        ["DSCF0656.JPG", 1227, 604, "TRAIN"],                  # kitchen frame 0
        ["DSCF0656.JPG", 645, 510, "TRAIN"],                   # kitchen frame 0
        ["DSCF0656.JPG", 963, 506, "TRAIN"],                   # kitchen frame 0
        ["DSCF0658.JPG", 1453, 301, "TEST"],                   # kitchen frame 2
        ["DSCF0658.JPG", 751, 172, "TEST"],                    # kitchen frame 2
        ["DSCF0658.JPG", 1236, 650, "TEST"],                   # kitchen frame 2
        ["DSCF0677.JPG", 639, 523, "TRAIN"],                   # kitchen frame 21
        ["DSCF0677.JPG", 846, 589, "TRAIN"],                   # kitchen frame 21
        ["DSCF0677.JPG", 1353, 117, "TRAIN"],                  # kitchen frame 21
        ["DSCF0686.JPG", 990, 68, "TEST"],                     # kitchen frame 30
        ["DSCF0686.JPG", 617, 191, "TEST"],                    # kitchen frame 30
        ["DSCF0686.JPG", 534, 456, "TEST"],                    # kitchen frame 30
        ["DSCF0737.JPG", 924, 437, "TRAIN"],                   # kitchen frame 81
        ["DSCF0737.JPG", 608, 691, "TRAIN"],                   # kitchen frame 81
        ["DSCF0738.JPG", 861, 206, "TEST"],                    # kitchen frame 82
        ["DSCF0738.JPG", 956, 436, "TEST"],                    # kitchen frame 82
        ["DSCF0738.JPG", 637, 721, "TEST"],                    # kitchen frame 82
        ["DSCF0775.JPG", 934, 169, "TRAIN"],                   # kitchen frame 119
        ["DSCF0775.JPG", 652, 232, "TRAIN"],                   # kitchen frame 119
        ["DSCF0779.JPG", 1076, 102, "TEST"],                   # kitchen frame 123
        ["DSCF0779.JPG", 139, 161, "TEST"],                    # kitchen frame 123
        ["DSCF0779.JPG", 886, 182, "TEST"],                    # kitchen frame 123
        ["DSCF0810.JPG", 694, 306, "TEST"],                    # kitchen frame 154
        ["DSCF0818.JPG", 1183, 54, "TRAIN"],                   # kitchen frame 162
        ["DSCF0820.JPG", 1258, 56, "TEST"],                    # kitchen frame 164
        ["DSCF0884.JPG", 802, 133, "TRAIN"],                   # kitchen frame 228
        ["DSCF0892.JPG", 1061, 114, "TEST"],                   # kitchen frame 236
        ["stove-gloves.jpeg", 41, 107, "EXTERNAL"],            # attach a Kaggle Dataset containing 'stove-gloves.jpeg'
        ["wooden-chair.webp", 422, 15, "EXTERNAL"],            # attach a Kaggle Dataset containing 'wooden-chair.webp'
        ["wooden-chair.webp", 309, 54, "EXTERNAL"],            # attach a Kaggle Dataset containing 'wooden-chair.webp'
        ["yellow-tractore-front.webp", 119, 282, "EXTERNAL"],  # attach a Kaggle Dataset containing 'yellow-tractore-front.webp'
        ["yellow-tractore-front.webp", 139, 42, "EXTERNAL"],   # attach a Kaggle Dataset containing 'yellow-tractore-front.webp'
        ["yellow-tractore-front.webp", 196, 203, "EXTERNAL"],  # attach a Kaggle Dataset containing 'yellow-tractore-front.webp'
        ["yellow-tractore-front.webp", 75, 199, "EXTERNAL"],   # attach a Kaggle Dataset containing 'yellow-tractore-front.webp'
        ["yellow-tractore-side.webp", 177, 172, "EXTERNAL"],   # attach a Kaggle Dataset containing 'yellow-tractore-side.webp'
        ["yellow-tractore-side.webp", 47, 186, "EXTERNAL"],    # attach a Kaggle Dataset containing 'yellow-tractore-side.webp'
    ],
    'room': [
        ["DSCF4667.JPG", 994, 259, "TRAIN"],                 # room frame 0
        ["DSCF4667.JPG", 441, 768, "TRAIN"],                 # room frame 0
        ["DSCF4667.JPG", 850, 677, "TRAIN"],                 # room frame 0
        ["DSCF4667.JPG", 1193, 29, "TRAIN"],                 # room frame 0
        ["DSCF4667.JPG", 1241, 619, "TRAIN"],                # room frame 0
        ["DSCF4667.JPG", 483, 300, "TRAIN"],                 # room frame 0
        ["DSCF4669.JPG", 33, 517, "TRAIN"],                  # room frame 2
        ["DSCF4676.JPG", 274, 302, "TEST"],                  # room frame 9
        ["DSCF4676.JPG", 1001, 547, "TEST"],                 # room frame 9
        ["DSCF4676.JPG", 983, 813, "TEST"],                  # room frame 9
        ["DSCF4686.JPG", 918, 385, "TEST"],                  # room frame 19
        ["DSCF4686.JPG", 134, 508, "TEST"],                  # room frame 19
        ["DSCF4686.JPG", 1105, 220, "TEST"],                 # room frame 19
        ["DSCF4698.JPG", 420, 306, "TRAIN"],                 # room frame 31
        ["DSCF4698.JPG", 316, 409, "TRAIN"],                 # room frame 31
        ["DSCF4698.JPG", 834, 291, "TRAIN"],                 # room frame 31
        ["DSCF4706.JPG", 494, 430, "TEST"],                  # room frame 39
        ["DSCF4706.JPG", 676, 307, "TEST"],                  # room frame 39
        ["DSCF4715.JPG", 110, 157, "TRAIN"],                 # room frame 48
        ["DSCF4716.JPG", 593, 128, "TEST"],                  # room frame 49
        ["DSCF4724.JPG", 169, 469, "TRAIN"],                 # room frame 57
        ["DSCF4736.JPG", 216, 500, "TEST"],                  # room frame 69
        ["DSCF4747.JPG", 184, 732, "TRAIN"],                 # room frame 80
        ["DSCF4747.JPG", 923, 991, "TRAIN"],                 # room frame 80
        ["DSCF4756.JPG", 189, 472, "TEST"],                  # room frame 89
        ["DSCF4780.JPG", 482, 370, "TRAIN"],                 # room frame 113
        ["DSCF4796.JPG", 337, 200, "TEST"],                  # room frame 129
        ["DSCF4797.JPG", 495, 284, "TRAIN"],                 # room frame 130
        ["DSCF4899.JPG", 767, 817, "TRAIN"],                 # room frame 232
        ["home theater system.jpg", 1258, 428, "EXTERNAL"],  # attach a Kaggle Dataset containing 'home theater system.jpg'
        ["home theater system.jpg", 636, 460, "EXTERNAL"],   # attach a Kaggle Dataset containing 'home theater system.jpg'
        ["metal bowe.jpg", 98, 112, "EXTERNAL"],             # attach a Kaggle Dataset containing 'metal bowe.jpg'
        ["piano.jpeg", 1139, 1015, "EXTERNAL"],              # attach a Kaggle Dataset containing 'piano.jpeg'
        ["television.jpg", 109, 79, "EXTERNAL"],             # attach a Kaggle Dataset containing 'television.jpg'
        ["television.jpg", 450, 35, "EXTERNAL"],             # attach a Kaggle Dataset containing 'television.jpg'
        ["white dor.jpeg", 159, 153, "EXTERNAL"],            # attach a Kaggle Dataset containing 'white dor.jpeg'
    ],
}
GRID_VIEW = {s: 0 for s in SCENES}   # frame to show large with a coordinate grid, per scene

_DS = globals().get('NERFACTO_DOWNSCALE', 2)   # same folder pick_points.py / cell 6b use

def _probe_tag_backoff(c):
    """Fields after [ref, x, y]: an optional 'TRAIN'/'TEST'/'EXTERNAL' tag, then an
    optional numeric backoff. A number in the tag's slot is the older backoff-only
    form. Same rule as explorer.py's parse_probe -- keep the two in step."""
    rest = list(c[3:])
    tag = rest.pop(0) if rest and isinstance(rest[0], str) else ''
    return tag, (float(rest[0]) if rest else -1.0)


def _circle(ax, x, y, k, wh):
    ax.add_patch(Circle((x, y), radius=max(wh) // 45, fill=False, color='red', lw=2))
    ax.text(x + 12, y - 12, str(k), color='red', fontsize=13, weight='bold')

json.dump({s: [list(c) for c in COORDS.get(s, [])] for s in SCENES},
          open(f'{WORK}/coords.json', 'w'))
print(f'wrote {WORK}/coords.json')

for s in SCENES:
    cs = COORDS.get(s, [])
    if not cs:
        print(f'--- {s}: no COORDS, skipping preview (and cell 6b) ---')
        continue

    _imgs = sorted(glob.glob(f'{SCENE_DATA(s)}/images_{_DS}/*'))
    assert _imgs, f'no images in {SCENE_DATA(s)}/images_{_DS}/ -- run cell 2b'
    _by_name = {os.path.basename(p): p for p in _imgs}

    def _resolve(ref, _imgs=_imgs, _by_name=_by_name, _scene=s):
        if isinstance(ref, int):
            assert 0 <= ref < len(_imgs), f'frame index {ref} out of range 0..{len(_imgs) - 1}'
            return _imgs[ref]
        if os.path.basename(ref) in _by_name:
            return _by_name[os.path.basename(ref)]
        hits = sorted(glob.glob(f'/kaggle/input/**/{os.path.basename(ref)}', recursive=True))
        if not hits:
            print(f'  ! {ref!r} is not a {_scene} frame and not under /kaggle/input --',
                  'attach a Kaggle Dataset containing it (cell 6b will fail without it)')
        return hits[0] if hits else None

    print(f'--- {s}: {len(_imgs)} frames ---')
    for _c in cs:
        _tag, _bk = _probe_tag_backoff(_c)
        _mode = f'manual {_bk}' if _bk > 0 else 'auto backoff'
        print('  ', _c[:3], f'[{_tag or "untagged"}, {_mode}]', '->', _resolve(_c[0]))

    _gv = GRID_VIEW.get(s, 0)
    _gp = _resolve(_gv)
    _im = Image.open(_gp); _w, _h = _im.size
    print(f'  grid view: {os.path.basename(str(_gp))} (W,H)=({_w},{_h})')
    plt.figure(figsize=(15, 10))
    plt.imshow(_im)
    plt.xticks(range(0, _w, 100)); plt.yticks(range(0, _h, 100))
    plt.grid(True, color='cyan', alpha=0.4, lw=0.5)
    for _k, _c in enumerate(cs):
        _ref, _x, _y = _c[0], _c[1], _c[2]
        if _resolve(_ref) == _gp:
            _circle(plt.gca(), _x, _y, _k, (_w, _h))
    plt.title(f'{s}: {os.path.basename(str(_gp))} -- read (x, y) off the grid; circles = COORDS here')
    plt.show()

    _seen, _panels = set(), []
    for _c in cs:
        _p = _resolve(_c[0])
        if _p and _p not in _seen:
            _seen.add(_p); _panels.append((_c[0], _p))
    if _panels:
        _ncol = min(len(_panels), 4)
        _nrow = math.ceil(len(_panels) / _ncol)
        _fig, _ax = plt.subplots(_nrow, _ncol, figsize=(5 * _ncol, 4 * _nrow), squeeze=False)
        for _slot in range(_nrow * _ncol):
            _a = _ax[_slot // _ncol][_slot % _ncol]; _a.axis('off')
            if _slot >= len(_panels):
                continue
            _ref, _path = _panels[_slot]
            _pim = Image.open(_path); _pw, _ph = _pim.size
            _a.imshow(_pim); _a.set_title(f'{s}: {os.path.basename(str(_path))}')
            for _k, _c in enumerate(cs):
                if _resolve(_c[0]) == _path:
                    _circle(_a, _c[1], _c[2], _k, (_pw, _ph))
        plt.show()

In [ ]:
EXPLORER_SRC = r'''"""Headless point-and-generate explorer for the conditional-NF fork.

This is the committed-run ("Save & Run All") path for probing the conditional NF:
it reproduces app/gradio_app.py's workflow without a UI. For each (ref, x, y)
probe -- ref = a dataset frame index or an external image's basename -- take that
pixel's DINOv2 patch-token feature as the NF condition, sample candidate (point,
direction) pairs, rank by likelihood, render the top few through the frozen NeRF,
and write a montage PNG: the source frame, then one panel per render showing
  * the render, washed green where VF-NeRF's view-likelihood mask keeps a pixel,
    with the wash's opacity graded by that pixel's likelihood (0.55 = most likely
    in the view, 0.2 = just above the mask threshold);
  * a red box on the DINO patch the sampled point lands in, whose feature is
    scored against the source feature by cosine similarity ("pt cos");
  * a green circle where the SOURCE pixel's own 3-D point (the frozen NeRF's
    depth along the source camera ray -- dataset frames only) projects into this
    view, and the 3-D distance between that point and the NF-sampled point.
(Points come from the notebook's cell 6a; run app/gradio_app.py directly on a
local GPU box for a live click UI.)

Runs inside the py3.10 venv (needs torch 1.13 + nerfstudio + tinycudann); the
notebook cell that launches it displays the PNGs from the base kernel.
"""
import argparse
import glob
import json
import math
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
import numpy as np
import torch
import torch.nn.functional as F

REPO = "/kaggle/temp/VF-NeRF-conditional"
sys.path.insert(0, REPO)

from nerfstudio.cameras.cameras import Cameras
from nerfstudio.fields.nf_field import ConditionalNFField
from nerfstudio.utils.dino_features import DinoExtractor, load_image_chw_01, patch_pixel_box
from nerfstudio.utils.eval_utils import eval_setup

# nerfstudio's default post-auto-orient world-up axis (orientation-method "up").
DEFAULT_WORLD_UP = torch.tensor([0.0, 0.0, 1.0])
# Source-panel title colour per COORDS tag, matching app/pick_points.py's window.
TAG_COLOR = {"TRAIN": "darkgreen", "TEST": "orangered"}
# Mask wash opacity range: a kept pixel at the mask threshold gets MASK_ALPHA_LO,
# the most likely pixel of the view gets MASK_ALPHA_HI.
MASK_ALPHA_LO, MASK_ALPHA_HI = 0.2, 0.55


# --- helpers copied from app/gradio_app.py (kept gradio-free) ------------------

def default_backoff_distance(cameras: Cameras) -> float:
    centers = cameras.camera_to_worlds[..., :3, 3]
    scene_center = centers.mean(dim=0)
    return (centers - scene_center).norm(dim=-1).median().item()


def build_camera_from_point_direction(position, direction, reference_cameras, backoff_distance,
                                      world_up=DEFAULT_WORLD_UP):
    device = position.device
    world_up = world_up.to(device)
    forward = direction / direction.norm().clamp_min(1e-8)
    up_ref = world_up
    if torch.abs(torch.dot(forward, up_ref)) > 0.99:
        up_ref = torch.tensor([1.0, 0.0, 0.0], device=device)
    right = torch.cross(forward, up_ref)
    right = right / right.norm().clamp_min(1e-8)
    up = torch.cross(right, forward)
    camera_origin = position - forward * backoff_distance
    rotation = torch.stack([right, up, -forward], dim=-1)
    c2w = torch.cat([rotation, camera_origin.unsqueeze(-1)], dim=-1)
    return Cameras(
        camera_to_worlds=c2w.unsqueeze(0),
        fx=reference_cameras.fx[0:1], fy=reference_cameras.fy[0:1],
        cx=reference_cameras.cx[0:1], cy=reference_cameras.cy[0:1],
        width=reference_cameras.width[0:1], height=reference_cameras.height[0:1],
        camera_type=reference_cameras.camera_type[0:1],
    ).to(device)


def project_point(cam: Cameras, P: torch.Tensor):
    """World point -> (x, y) pixel of the one-camera `cam`, plus whether it lies in
    front of the camera and its Euclidean distance from the camera origin.

    Exact inverse of Cameras._generate_rays_from_coords: camera frame is OpenGL
    (local -Z forward, +Y up, +X right) and image y grows downward, so
        p_cam = R^T (P - t);  z = -p_cam.z
        x = cx + fx * p_cam.x / z;   y = cy - fy * p_cam.y / z
    Intrinsics are read from `cam` as it is, so call this AFTER any
    rescale_output_resolution. Lens distortion is ignored (the mip-NeRF 360
    reconstructions carry none worth a pixel).
    """
    c2w = cam.camera_to_worlds[0]
    R, t = c2w[:, :3], c2w[:, 3]
    pc = R.T @ (P - t)
    z = float(-pc[2])
    dist = float(pc.norm())
    if z <= 1e-6:
        return math.nan, math.nan, False, dist
    x = float(cam.cx[0, 0]) + float(cam.fx[0, 0]) * float(pc[0]) / z
    y = float(cam.cy[0, 0]) - float(cam.fy[0, 0]) * float(pc[1]) / z
    return x, y, True, dist


def load_conditional_nf(checkpoint_path: Path, device: torch.device):
    ckpt = torch.load(checkpoint_path, map_location=device)
    field = ConditionalNFField(
        context_dim=ckpt["context_dim"],
        num_dims=ckpt.get("num_dims", 6),
        num_blocks=ckpt["num_blocks"],
        hidden_dim=ckpt["hidden_dim"],
        cond_prior=ckpt["cond_prior"],
        use_cond_in_coupling=True,
        use_batchnorm=ckpt["use_batchnorm"],
        reduce_dim=ckpt.get("reduce_dim"),
        reduce_divide_factor=ckpt.get("reduce_divide_factor", 8),
        device=str(device),
    )
    field.load_state_dict(ckpt["model_state"])
    field.eval()
    return field, ckpt["dino_model_name"], int(ckpt.get("step", -1))


def mask_alpha(score_hw, mask_hw, threshold, alpha_lo=MASK_ALPHA_LO, alpha_hi=MASK_ALPHA_HI):
    """Per-pixel wash opacity: 0 where the mask rejects, else a linear ramp from
    `alpha_lo` at the mask threshold to `alpha_hi` at the view's most likely pixel
    (`score` is already min-max normalised per view, so its max is ~1)."""
    span = max(1.0 - threshold, 1e-6)
    ramp = np.clip((score_hw - threshold) / span, 0.0, 1.0)
    return np.where(mask_hw, alpha_lo + (alpha_hi - alpha_lo) * ramp, 0.0).astype(np.float32)


def draw_mask_overlay(ax, rgb_hwc, score_hw, mask_hw, threshold):
    """The render with every PIXEL that PASSES the view-likelihood mask washed
    green, more opaque the more likely the pixel (see `mask_alpha`).

    `score_hw` / `mask_hw` are the render's own [H, W] grids, so the overlay needs
    no extent, no interpolation and no clipping -- it lands on the image one-to-one.
    """
    rgba = np.zeros(mask_hw.shape + (4,), dtype=np.float32)
    rgba[..., 1] = 1.0                                # green
    rgba[..., 3] = mask_alpha(score_hw, mask_hw, threshold)
    ax.imshow(rgb_hwc)
    ax.imshow(rgba)
    ax.axis("off")


def draw_patch_inset(ax, rgb_hwc, x0, y0, sx, sy, px=None, py=None, zoom=15.0, frac=0.26):
    """Corner inset repeating the patch box magnified, so a 14x14 cell on a 1559px
    frame is findable. The main panel keeps the box at true scale -- this only makes
    it visible, it does not restate its size.

    `(px, py)`, when given, is marked with a small cross: the pixel that was actually
    probed, which `patch_cell`'s binning does not always place inside the cell it
    reads (see `patch_box_at`). Showing both makes that offset legible instead of
    looking like a misdrawn box.
    """
    h, w = rgb_hwc.shape[:2]
    zoom = max(1.0, min(zoom, w / sx, h / sy))       # never crop wider than the image
    half_w, half_h = sx * zoom / 2, sy * zoom / 2
    cx = min(max(x0 + sx / 2, half_w), w - half_w)
    cy = min(max(y0 + sy / 2, half_h), h - half_h)
    ins = ax.inset_axes([1 - frac - 0.02, 1 - frac - 0.02, frac, frac])
    ins.imshow(rgb_hwc)
    ins.set_xlim(cx - half_w, cx + half_w)
    ins.set_ylim(cy + half_h, cy - half_h)
    ins.add_patch(Rectangle((x0, y0), sx, sy, fill=False, color="red", lw=1.2))
    if px is not None:
        ins.plot([px], [py], marker="+", color="red", markersize=5, markeredgewidth=0.9)
    ins.set_xticks([])
    ins.set_yticks([])
    for spine in ins.spines.values():
        spine.set_edgecolor("red")
        spine.set_linewidth(0.8)


def parse_probe(entry):
    """COORDS row -> (ref, x, y, tag, backoff_override).

    Accepts [ref, x, y], [ref, x, y, tag], [ref, x, y, tag, backoff] and the older
    [ref, x, y, backoff]: a string in the 4th slot is the picker's TRAIN/TEST/
    EXTERNAL tag, a number there is a backoff. The tag is provenance only -- the
    split is already baked into the frozen NeRF this runs against, and a frame's
    camera is found by name in either split, not by reading the tag.
    Cell 6a's _probe_tag_backoff applies the same rule; keep the two in step.
    """
    rest = list(entry[3:])
    tag = rest.pop(0) if rest and isinstance(rest[0], str) else ""
    return entry[0], int(entry[1]), int(entry[2]), tag, (float(rest[0]) if rest else -1.0)


# --- main --------------------------------------------------------------------

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--nerf-config", required=True)
    ap.add_argument("--scene-dir", required=True)
    ap.add_argument("--cond-nf-checkpoint", required=True)
    ap.add_argument("--probes", required=True,
                    help='JSON list of [ref, x, y], [ref, x, y, tag] or [ref, x, y, tag, '
                         'backoff]; ref = int frame index or "name.ext" external image. tag = '
                         'the picker TRAIN/TEST/EXTERNAL label, echoed to the montage and to '
                         'dino_consistency.json. backoff <= 0 (or omitted) = auto (NeRF depth '
                         'for dataset frames, scene constant otherwise); > 0 = that literal.')
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--downscale", type=int, default=2,
                    help="images_<N>/ folder the coords were picked on (must match cell 6a / pick_points.py)")
    ap.add_argument("--num-samples", type=int, default=200)
    ap.add_argument("--render-top", type=int, default=3)
    ap.add_argument("--render-downscale", type=float, default=3.0)
    ap.add_argument("--vf-score-threshold", type=float, default=0.05,
                    help="VF-NeRF view-likelihood mask threshold on the min-max "
                         "normalised per-PIXEL likelihood (the original's default). "
                         "0 disables the mask -- every pixel survives.")
    ap.add_argument("--vf-chunk", type=int, default=1 << 15,
                    help="rows per conditional-flow batch when scoring the per-pixel "
                         "view likelihood (~180k pixels per view); lower it if a "
                         "smaller GPU runs out of memory")
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Loading frozen NeRF from {args.nerf_config} ...", flush=True)
    config, pipeline, _, _ = eval_setup(Path(args.nerf_config), test_mode="inference")
    nerf_model = pipeline.model.to(device).eval()
    for p in nerf_model.parameters():
        p.requires_grad_(False)

    # Build the reference cameras in the SAME frame the frozen NeRF (and hence the
    # conditional NF) was trained in -- the checkpoint's own dataparser config,
    # not stock defaults (center-method / scene-scale differ). Both splits: a
    # held-out (TEST) frame has a real camera pose too, only the NeRF and the flow
    # never saw its pixels -- so its ray still gives the true source point.
    dp = config.pipeline.datamanager.dataparser
    dp.data = Path(args.scene_dir)
    outs = dp.setup().get_dataparser_outputs(split="train")
    cameras = outs.cameras.to(device)
    # name -> (cameras, index within that split, split); dataparser order per
    # split (matches each `cameras`), NOT sorted(glob(...)) order.
    cam_by_name = {Path(p).name: (cameras, i, "train") for i, p in enumerate(outs.image_filenames)}
    try:
        outs_te = dp.setup().get_dataparser_outputs(split="test")
        cameras_te = outs_te.cameras.to(device)
        for i, p in enumerate(outs_te.image_filenames):
            cam_by_name.setdefault(Path(p).name, (cameras_te, i, "test"))
        n_test = len(outs_te.image_filenames)
    except Exception as e:  # e.g. train_split_fraction=1.0 leaves no held-out frames
        print(f"  ! no test-split cameras ({e!r}); held-out frames fall back to the constant backoff", flush=True)
        n_test = 0
    const_backoff = default_backoff_distance(cameras)
    # normalized-world -> original-dataset units: divide a distance by this.
    dp_scale = float(getattr(outs, "dataparser_scale", 1.0) or 1.0)

    # Image list + coordinate space = sorted(glob("images_<downscale>/*")), the
    # EXACT folder cell 6a and pick_points.py pick on (--downscale is passed from
    # cell 6b, not guessed from the nerfacto config, so it can't drift). The
    # conditional NF was also trained on DINO features from this same folder.
    img_dir = Path(args.scene_dir) / f"images_{args.downscale}"
    image_filenames = sorted(img_dir.glob("*"))
    assert image_filenames, f"no images in {img_dir} -- run cell 1 (or fix --downscale)"
    print(f"{len(image_filenames)} images in {img_dir} | cameras: {len(outs.image_filenames)} train"
          f" + {n_test} test | const backoff {const_backoff:.3f}", flush=True)

    field, dino_model_name, step = load_conditional_nf(Path(args.cond_nf_checkpoint), device)
    print(f"conditional-NF checkpoint step = {step}", flush=True)
    extractor = DinoExtractor(model_name=dino_model_name, device=str(device))

    grid_cache = {}

    by_name = {p.name: p for p in image_filenames}

    def resolve_ref(ref):
        """ref -> image path.
          * int / digit string -> dataset frame by position (old form)
          * a dataset frame's basename -> that file in images_<ds>/
          * any other basename / path -> an external image: a real path if it
            exists, else looked up under /kaggle/input/** (attach a Dataset)."""
        if isinstance(ref, int) or (isinstance(ref, str) and ref.lstrip("-").isdigit()):
            i = int(ref)
            assert 0 <= i < len(image_filenames), (
                f"frame index {i} out of range 0..{len(image_filenames) - 1}")
            return image_filenames[i]
        p = Path(ref)
        if p.name in by_name:
            return by_name[p.name]
        if p.is_file():
            return p
        hits = sorted(glob.glob(f"/kaggle/input/**/{p.name}", recursive=True))
        if not hits:
            raise RuntimeError(
                f"image {ref!r} not found -- it is not a dataset frame (images/{p.name}) "
                f"and not under /kaggle/input/**; attach a Kaggle Dataset that contains "
                f"{p.name!r} (right sidebar -> Input -> Add Input)")
        return Path(hits[0])

    def patch_cell(grid, h, w, x, y):
        """Nearest patch-grid cell for pixel (x, y) of an (h, w) image -> [EMBED_DIM]."""
        hp, wp = grid.shape[-2:]
        py = min(max(int(y * hp / h), 0), hp - 1)
        px = min(max(int(x * wp / w), 0), wp - 1)
        return grid[:, py, px].reshape(-1)

    def patch_box_at(grid, h, w, x, y):
        """Pixel (x, y) -> (x0, y0, sx, sy) of the patch cell it falls in, in that
        image's own pixels.

        Binning identical to `patch_cell` (which defines the condition lookup), box
        geometry from `patch_pixel_box`, so the drawn rectangle is exactly the cell
        whose feature was read -- 14x14 natively, PATCH_SIZE/scale for an image past
        MAX_DINO_SIDE.

        Note the two do not always agree: `patch_cell` bins with int(x * wp / w),
        spreading the image over the PADDED grid (wp * side >= w), while the cells
        themselves tile in exact `side` steps. For ~30% of columns on a 1559px frame
        the cell read is the neighbour of the one containing the pixel, by up to 8px
        (11px on a 520px render). This draws the cell actually read, because that is
        where the reported feature came from; `draw_patch_inset` also marks the
        probed pixel so the offset is visible. `patch_cell` is deliberately left
        alone -- it must stay identical to the NF trainer's sample_batch.
        """
        hp, wp = grid.shape[-2:]
        py = min(max(int(y * hp / h), 0), hp - 1)
        px = min(max(int(x * wp / w), 0), wp - 1)
        return patch_pixel_box(py, px, h, w)

    def view_likelihood_mask(rays, depth, cond, threshold, chunk):
        """VF-NeRF's view-likelihood mask for one rendered view, PER PIXEL.

        The original (nerfacto.py:392-409 @ c76d3bd) computed, per pixel:
            P = origins + directions * depth        # NeRF-rendered surface point
            view_log_likelihood = nf_model.log_prob(cat([P, directions]))
        i.e. the value came from the normalizing flow, the NeRF supplied only the
        depth. This fork's nerfacto has no such head (stripped with the registration
        pipeline), so the same quantity is rebuilt from the conditional flow, scored
        under the SAME `cond` that generated this view: the map then reads "where
        does the flow believe the queried feature can be seen from here".

        `rays` and `depth` are the bundle and the [H, W, 1] depth map the view was
        already rendered from, so every pixel's P and direction are in hand and this
        costs no extra NeRF work -- only the flow, chunked at `chunk` rows because
        H*W is ~180k for a 519x346 render. BatchNormFlow uses running statistics in
        eval(), so the chunking is exact and batch-size independent.

        Returns (score[H*W], mask[H*W], degenerate, ll[H*W]). `ll` is the
        finite-repaired log-likelihood the mask was derived from, so reporting its
        range over the kept pixels describes exactly what the threshold cut.
        """
        P = (rays.origins + rays.directions * depth).reshape(-1, 3)
        x = torch.cat([P, rays.directions.reshape(-1, 3)], dim=-1)      # [H*W, 6]
        with torch.no_grad():
            ll = torch.cat([
                field.log_prob(x[i:i + chunk],
                               cond.unsqueeze(0).expand(len(x[i:i + chunk]), -1)
                               ).reshape(-1)
                for i in range(0, len(x), chunk)])                      # [H*W]

        # get_mask_from_view_likelihood (exporter_utils.py:595-632 @ c76d3bd):
        # non-finite -> min, exp, min-max normalise, >= threshold. Normalised per
        # rendered view, so each view keeps its own brightest region.
        finite = torch.isfinite(ll)
        if finite.any():
            ll = torch.where(finite, ll, ll[finite].min())
        else:
            ll = torch.zeros_like(ll)
        # Stable rewrite of (exp(ll) - exp(ll).min()) / exp(ll).max(): factor out
        # exp(max) so a large log-prob cannot overflow to inf before the division.
        # Algebraically identical, including the max of 1 - exp(min - max).
        m = ll.max()
        score = torch.exp(ll - m) - torch.exp(ll.min() - m)
        mask = score >= threshold
        # The max sits at ~1.0, so an empty mask means a constant / all-non-finite
        # map. Fall back to every pixel and flag it rather than dropping the view.
        degenerate = not bool(mask.any())
        if degenerate:
            mask = torch.ones_like(mask)
        return score, mask, degenerate, ll

    def patch_feature(ref, x, y):
        """Pixel (x, y) on image `ref` -> its DINOv2 patch-token feature.

        Indexes the patch grid directly (same as the NF trainer's sample_batch);
        no giant per-pixel upsampled map.
        """
        if ref not in grid_cache:
            img = load_image_chw_01(resolve_ref(ref))
            with torch.no_grad():
                g, (h, w) = extractor.extract_patch_grid(img)
            grid_cache[ref] = (g.float(), img, h, w)
        g, img, h, w = grid_cache[ref]
        return patch_cell(g, h, w, x, y).to(device), img

    def probe_backoff(ref, x, y, img_h, img_w):
        """Camera standoff for rendering this probe's candidates, and the probe's
        own 3-D point.

        A dataset frame (either split) has a real camera ray through pixel (x, y);
        the frozen NeRF's median termination depth along it is the true
        camera-to-surface distance -- a far better render backoff than the
        scene-wide constant -- and origin + direction * depth is the SOURCE POINT
        S the sampled candidates should land near. External images have no camera
        in this frame -> the constant, and no S.
        Returns (distance, "depth" | "depth(test)" | "const", S | None).
        """
        if isinstance(ref, int) or (isinstance(ref, str) and ref.lstrip("-").isdigit()):
            name = image_filenames[int(ref)].name
        else:
            name = Path(ref).name
        hit = cam_by_name.get(name)
        if hit is None:
            return const_backoff, "const", None
        cams, i, split = hit
        cy = y * float(cams.height[i]) / img_h
        cx = x * float(cams.width[i]) / img_w
        rb = cams.generate_rays(
            camera_indices=torch.tensor([[i]]),
            coords=torch.tensor([[cy, cx]], dtype=torch.float32),
        ).to(device)
        with torch.no_grad():
            t = float(nerf_model(rb)["depth"].reshape(-1)[0])
        if not math.isfinite(t) or t <= 1e-4:
            print(f"  ! depth render at ({x},{y}) = {t!r}; using constant backoff", flush=True)
            return const_backoff, "const", None
        S = rb.origins.reshape(-1, 3)[0] + rb.directions.reshape(-1, 3)[0] * t
        return t, ("depth" if split == "train" else "depth(test)"), S

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    probes = json.loads(args.probes)
    manifest = []
    records = []

    for k, entry in enumerate(probes):
        ref, x, y, tag, depth_override = parse_probe(entry)
        src = resolve_ref(ref)
        cond, img = patch_feature(ref, x, y)
        _H, _W = int(img.shape[-2]), int(img.shape[-1])
        bo, bo_src, S = probe_backoff(ref, x, y, _H, _W)
        if depth_override > 0:
            bo, bo_src = depth_override, "manual"
        print(f"probe {k}: {ref}{f' [{tag}]' if tag else ''} @ ({x},{y})  <- {src}  "
              f"({_W}x{_H})  backoff {bo:.3f} ({bo_src})"
              f"{'' if S is not None else '  [no camera -> no source point]'}", flush=True)
        if not (0 <= x < _W and 0 <= y < _H):
            print(f"  ! ({x},{y}) is OUTSIDE this {_W}x{_H} image -- coords were picked "
                  f"on a different resolution or a different frame", flush=True)
        with torch.no_grad():
            samples = field.sample(num_samples=args.num_samples, context=cond)
            logp = field.log_prob(
                samples, cond.unsqueeze(0).expand(args.num_samples, -1)
            ).squeeze(-1)
        order = torch.argsort(logp, descending=True)[: args.render_top]
        n = len(order)
        view_stats = []

        # One row: the source frame, then the top-n renders, each with the graded
        # mask wash, the sampled point's patch box and the projected source point.
        fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 4.9), squeeze=False)
        ax = axes[0]
        g_src, _, h_src, w_src = grid_cache[ref]
        src_rgb = img.permute(1, 2, 0).numpy()
        sx0, sy0, ssx, ssy = patch_box_at(g_src, h_src, w_src, x, y)
        ax[0].imshow(src_rgb)
        # The one patch cell `cond` was read from, at true scale -- ~3 screen pixels
        # on a 1559px frame, hence the inset.
        ax[0].add_patch(Rectangle((sx0, sy0), ssx, ssy, fill=False, color="red", lw=1.2))
        ax[0].axis("off")
        draw_patch_inset(ax[0], src_rgb, sx0, sy0, ssx, ssy, x, y)
        for j, idx in enumerate(order):
            cam = build_camera_from_point_direction(
                samples[idx, :3], samples[idx, 3:], cameras, bo
            )
            cam.rescale_output_resolution(1.0 / args.render_downscale)
            cx, cy = float(cam.cx.squeeze()), float(cam.cy.squeeze())
            rb_full = cam.generate_rays(camera_indices=0)
            with torch.no_grad():
                o = nerf_model.get_outputs_for_camera_ray_bundle(rb_full)
            # SAME extractor.extract_patch_grid path (pad / norm / downscale) used to
            # build `cond`.
            nv_chw = o["rgb"].clamp(0, 1).permute(2, 0, 1).contiguous().cpu()
            nv_rgb = nv_chw.permute(1, 2, 0).numpy()   # same pixels, plot-ready
            with torch.no_grad():
                g_nv, (h_nv, w_nv) = extractor.extract_patch_grid(nv_chw)
            # o["depth"] is the [H, W, 1] map this very render produced, and rb_full
            # carries that render's per-pixel origins/directions -- so the mask is
            # pixel-aligned with the image by construction, no resampling anywhere.
            vf_score, vf_mask, vf_degenerate, vf_ll = view_likelihood_mask(
                rb_full, o["depth"], cond, args.vf_score_threshold, args.vf_chunk)
            masked_frac = float(vf_mask.float().mean())
            # (a) DINO patch feature at the sampled point itself: P projects to
            # this view's principal point (cx, cy) -- the red box -- because the
            # camera is built pointing straight at P. Cosine similarity of that
            # patch to the source condition = "does what we aimed at look right".
            point_feat = patch_cell(g_nv, nv_chw.shape[-2], nv_chw.shape[-1], cx, cy).float().to(device)
            point_dino_cos = float(F.cosine_similarity(point_feat, cond, dim=0))
            # (b) the source pixel's own 3-D point S, projected into this view, and
            # its distance to the NF-sampled point P (same normalized frame).
            if S is not None:
                sx, sy, in_front, s_dist = project_point(cam, S)
                src_in_view = bool(in_front and 0 <= sx < w_nv and 0 <= sy < h_nv)
                # occluded from here if the render's own surface along that pixel
                # sits well in front of S (5% slack for the median-depth estimate)
                src_occluded = bool(src_in_view and
                                    float(o["depth"][int(sy), int(sx), 0]) < 0.95 * s_dist)
                sd_norm = float(torch.linalg.norm(S - samples[idx, :3]))
                sd_orig = sd_norm / dp_scale
            else:
                sx = sy = math.nan
                src_in_view = src_occluded = False
                sd_norm = sd_orig = math.nan
            ll_kept = vf_ll[vf_mask]
            ll_lo, ll_hi = float(ll_kept.min()), float(ll_kept.max())
            ll_all_lo, ll_all_hi = float(vf_ll.min()), float(vf_ll.max())
            view_stats.append({"logp": float(logp[idx].item()),
                               "point_dino_cos": point_dino_cos,
                               "src_dist_norm": sd_norm, "src_dist_orig": sd_orig,
                               "src_in_view": src_in_view, "src_occluded": src_occluded,
                               "src_px": [sx, sy],
                               "masked_frac": masked_frac,
                               "ll_kept_min": ll_lo, "ll_kept_max": ll_hi,
                               "ll_min": ll_all_lo, "ll_max": ll_all_hi,
                               "mask_degenerate": vf_degenerate})
            # Render + graded mask wash + the red patch box + the green source circle,
            # all on one image.
            draw_mask_overlay(ax[j + 1], nv_rgb,
                              vf_score.reshape(h_nv, w_nv).cpu().numpy(),
                              vf_mask.reshape(h_nv, w_nv).cpu().numpy(),
                              args.vf_score_threshold)
            px0, py0, psx, psy = patch_box_at(g_nv, h_nv, w_nv, cx, cy)
            ax[j + 1].add_patch(Rectangle((px0, py0), psx, psy,
                                          fill=False, color="red", lw=0.9))
            if src_in_view:
                ax[j + 1].add_patch(Circle((sx, sy), radius=max(3.0, 0.012 * min(h_nv, w_nv)),
                                           fill=False, color="lime", lw=1.5,
                                           ls="--" if src_occluded else "-"))
            if S is None:
                src_line = "src pt n/a (no camera)"
            elif not src_in_view:
                src_line = f"src pt off-view  |S-P| {sd_norm:.3f} / {sd_orig:.3f}"
            else:
                src_line = (f"|S-P| {sd_norm:.3f} / {sd_orig:.3f}"
                            f"{' (occluded)' if src_occluded else ''}")
            ax[j + 1].set_title(f"logp {logp[idx].item():.2f}   pt cos {point_dino_cos:.3f}\n"
                                f"{src_line}\n"
                                f"mask kept {100 * masked_frac:.0f}%"
                                f"{' (degenerate)' if vf_degenerate else ''}"
                                f"  ll kept [{ll_lo:.1f}, {ll_hi:.1f}]", fontsize=9)
            print(f"  view {j}: logp {logp[idx].item():.2f}  pt cos {point_dino_cos:.3f}  "
                  f"{src_line}  mask {100 * masked_frac:.0f}%  ll kept [{ll_lo:.2f}, {ll_hi:.2f}]",
                  flush=True)

        _m = lambda key, rows: (sum(r[key] for r in rows) / len(rows)) if rows else math.nan
        mean_point_cos = _m("point_dino_cos", view_stats)
        mean_sd_norm = _m("src_dist_norm", view_stats) if S is not None else math.nan
        mean_sd_orig = _m("src_dist_orig", view_stats) if S is not None else math.nan
        nv = len(view_stats)
        ax[0].set_title(f"{ref}{f' [{tag}]' if tag else ''} @ ({x},{y})\n"
                        f"backoff {bo:.3f} ({bo_src})\n"
                        f"mean of {nv} view{'' if nv == 1 else 's'}:  pt cos {mean_point_cos:.3f}\n"
                        + (f"|S-P| {mean_sd_norm:.3f} / {mean_sd_orig:.3f}" if S is not None
                           else "no source point (no camera)"),
                        color=TAG_COLOR.get(tag.upper(), "black"), fontsize=9)
        fig.text(0.005, 0.01,
                 "red box   DINO patch the sampled point P lands in (its feature -> pt cos vs the source patch)\n"
                 "green o   the source pixel's own 3-D point S projected into the view (dashed = occluded there); "
                 "|S-P| in normalized / original units\n"
                 "green wash pixels passing the view-likelihood mask, opacity 0.2 -> 0.55 with the flow's "
                 "log-density (ll) at that pixel",
                 ha="left", va="bottom", fontsize=7.5, family="monospace", linespacing=1.3)

        png = out_dir / f"probe_{k:02d}.png"
        fig.tight_layout(rect=(0, 0.09, 1, 1))
        fig.savefig(png, dpi=90)
        plt.close(fig)
        manifest.append(str(png))
        records.append({"png": str(png), "probe": k, "ref": ref, "x": x, "y": y,
                        "tag": tag,
                        "backoff": bo, "backoff_src": bo_src,
                        "has_src_point": S is not None,
                        "src_point": S.tolist() if S is not None else None,
                        "mean_point_dino_cos": mean_point_cos,
                        "mean_src_dist_norm": mean_sd_norm,
                        "mean_src_dist_orig": mean_sd_orig,
                        "views": view_stats})
        print(f"probe {k}: {ref} ({x},{y}) -> {png}  "
              f"mean pt cos {mean_point_cos:.3f}  "
              f"mean |S-P| {mean_sd_norm:.3f} / {mean_sd_orig:.3f}", flush=True)

    all_point_cos = [v["point_dino_cos"] for r in records for v in r["views"]]
    with_src = [r for r in records if r["has_src_point"]]
    all_sd_norm = [v["src_dist_norm"] for r in with_src for v in r["views"]]
    all_sd_orig = [v["src_dist_orig"] for r in with_src for v in r["views"]]
    all_in_view = [v["src_in_view"] for r in with_src for v in r["views"]]
    _mean = lambda xs: sum(xs) / len(xs) if xs else math.nan
    summary = {
        "scene_mean_point_dino_cos": _mean(all_point_cos),
        "scene_mean_src_dist_norm": _mean(all_sd_norm),
        "scene_mean_src_dist_orig": _mean(all_sd_orig),
        "scene_frac_src_in_view": _mean([1.0 if v else 0.0 for v in all_in_view]),
        "n_probes_with_src": len(with_src),
        "dataparser_scale": dp_scale,
        "num_probes": len(records),
        "views_per_probe": args.render_top,
        "render_downscale": args.render_downscale,
        "vf_score_threshold": args.vf_score_threshold,
        "mask_alpha_range": [MASK_ALPHA_LO, MASK_ALPHA_HI],
        "scene_mean_masked_frac": _mean([v["masked_frac"] for r in records for v in r["views"]]),
        "probes": records,
    }
    (out_dir / "dino_consistency.json").write_text(json.dumps(summary, indent=2))
    print(f"scene mean: pt DINO cos {summary['scene_mean_point_dino_cos']:.3f}  "
          f"|S-P| {summary['scene_mean_src_dist_norm']:.3f} / "
          f"{summary['scene_mean_src_dist_orig']:.3f} over {len(with_src)} probes with a source point  -> "
          f"{out_dir / 'dino_consistency.json'}", flush=True)
    print("MANIFEST " + json.dumps(manifest), flush=True)


if __name__ == "__main__":
    main()
'''

In [ ]:
# @title 6b. Generate novel views for the picked points (per scene)
# For each scene with COORDS entries: for each point, take that pixel's DINOv2
# patch feature as the NF condition, draw NUM_SAMPLES candidate (position,
# direction) 6-D samples from that scene's flow, score each with the flow's
# log_prob, and render the RENDER_TOP HIGHEST-likelihood ones through that
# scene's frozen NeRF into a one-row montage: the source frame (red box = the one
# 14x14 DINO patch the condition was read from, magnified in the corner inset),
# then each view with three things drawn on the same image:
#   * green wash  = pixels passing VF-NeRF's view-likelihood mask (the flow's
#     log-density at each pixel's NeRF-depth surface point, exp'd, min-max
#     normalised per view, cut at VF_SCORE_THRESHOLD; 0 keeps every pixel). The
#     wash's opacity ramps 0.2 -> 0.55 with that likelihood;
#   * red box     = the patch the sampled point P projects into; its DINO feature's
#     cosine similarity to the source patch is the panel's 'pt cos' (read it
#     against cell 6c's same-patch / random-patch cosines for the scene);
#   * green circle = the SOURCE pixel's own 3-D point S (frozen-NeRF depth along
#     the source camera ray -- dataset frames of either split; external images
#     have no camera) projected into the view, dashed if the render occludes it;
#     '|S-P|' is the 3-D distance between S and the sampled P, normalized /
#     original units.
# A COORDS ref is a dataset frame's filename, an external image's filename
# (resolved under /kaggle/input/**), or an int index. Re-run after editing 6a;
# works in a committed run too. The log and each montage's left panel show
# 'backoff <v> (depth|depth(test)|const|manual)' -- the render-camera standoff:
# NeRF depth for a dataset-frame pixel (TEST frames now use their own held-out
# camera), that scene's constant otherwise, or a COORDS backoff override. Each
# probe's TRAIN/TEST/EXTERNAL tag from cell 6a is echoed in the log, on the
# montage, and into dino_consistency.json.
import subprocess, os, json, glob
from IPython.display import Image as _Img, display

COORDS = json.load(open(f'{WORK}/coords.json'))
assert any(COORDS.get(s) for s in SCENES), 'no points -- run cell 6a first'
print('points:', COORDS)
NUM_SAMPLES, RENDER_TOP, RENDER_DOWNSCALE = 1000, 3, 3.0
VF_SCORE_THRESHOLD = 0.05

subprocess.run([f'{VENV}/pip', 'install', '-q', 'matplotlib'], check=True)
with open('/kaggle/temp/explorer.py', 'w') as fh:
    fh.write(EXPLORER_SRC)

EXPLORE_DIR = f'{WORK}/explorer'
os.makedirs(EXPLORE_DIR, exist_ok=True)

for s in SCENES:
    probes = COORDS.get(s, [])
    if not probes:
        print(f'--- {s}: no COORDS, skipping ---')
        continue
    out_dir = f'{EXPLORE_DIR}/{s}'
    os.makedirs(out_dir, exist_ok=True)
    for _p in glob.glob(f'{out_dir}/probe_*.png'):
        os.remove(_p)
    cmd = [f'{VENV}/python', '-u', '/kaggle/temp/explorer.py',
           '--nerf-config', NERF_CONFIGS[s], '--scene-dir', SCENE_DATA(s),
           '--cond-nf-checkpoint', COND_LATEST[s],
           '--probes', json.dumps(probes), '--out-dir', out_dir,
           '--downscale', str(NERFACTO_DOWNSCALE),
           '--num-samples', str(NUM_SAMPLES), '--render-top', str(RENDER_TOP),
           '--render-downscale', str(RENDER_DOWNSCALE),
           '--vf-score-threshold', str(VF_SCORE_THRESHOLD)]
    print(f'--- {s} ---'); print(' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=REPO,
                   env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for png in sorted(glob.glob(f'{out_dir}/probe_*.png')):
        print(png)
        display(_Img(filename=png))
    _dj = f'{out_dir}/dino_consistency.json'
    if os.path.exists(_dj):
        _d = json.load(open(_dj))
        print(f"{s}: scene mean pt DINO cos {_d['scene_mean_point_dino_cos']:.3f}  "
              f"|S-P| {_d['scene_mean_src_dist_norm']:.3f} / {_d['scene_mean_src_dist_orig']:.3f} "
              f"over {_d['n_probes_with_src']} of {_d['num_probes']} probes with a source point "
              f"(source in view {100 * _d['scene_frac_src_in_view']:.0f}% of renders; "
              f"mask kept {100 * _d['scene_mean_masked_frac']:.0f}% of pixels)")
zip_report('report_explorer_probes', EXPLORE_DIR)

In [ ]:
# @title 6c. DINO reconstruction fidelity of the frozen NeRFs: cosine + random-patch floor (per scene)
# Everything above assumes a NeRF render is DINO-comparable to the real photo.
# This measures it: render each of MAX_FRAMES evenly-spaced frames per split from
# its own camera, extract the DINOv2 patch grid of the render and of the real
# image, and take the per-patch cosine similarity ("same-patch cos" -- the
# ceiling any novel-view score can reach). The FLOOR comes from the same frame:
# each real patch against a random (shuffled) render patch -- what two unrelated
# patches of this scene score by chance. 'separation' = ceiling - floor is the
# scale to read the explorer's 'pt cos' on; 'same>random' is the fraction of
# patches where the correct patch beats the random one.
import subprocess, os, json
RECON_DIR = f'{WORK}/dino_recon'
os.makedirs(RECON_DIR, exist_ok=True)
MAX_FRAMES = 40

DINO_RECON = {}
for s in SCENES:
    out = f'{RECON_DIR}/{s}_dino_recon.json'
    cmd = [f'{VENV}/python', '-u', 'scripts/eval_dino_reconstruction.py',
           '--nerf-config', NERF_CONFIGS[s], '--scene-dir', SCENE_DATA(s),
           '--output-path', out, '--max-frames', str(MAX_FRAMES)]
    print(f'--- {s} ---'); print(' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=REPO, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    DINO_RECON[s] = json.load(open(out))

hdr = (f"{'scene':<10}{'split':<7}{'| same cos':>11}{'var':>9}{'| random cos':>13}{'var':>9}"
       f"{'| separation':>13}{'same>rand%':>11}{'frames':>8}")
print(hdr); print('-' * len(hdr))
for s, d in DINO_RECON.items():
    for split, r in d['splits'].items():
        print(f"{s:<10}{split:<7}{r['mean_cos']:>11.4f}{r['var_cos']:>9.5f}"
              f"{r['mean_random_cos']:>13.4f}{r['var_random_cos']:>9.5f}"
              f"{r['separation']:>13.4f}{100 * r['mean_frac_same_beats_random']:>11.1f}{r['n_frames']:>8d}")
    if 'test_minus_train' in d:
        print(f"{'':<10}{'gap':<7}{d['test_minus_train']:>11.4f}   (test - train, same-patch cos)")
zip_report('report_dino_recon', RECON_DIR)

In [ ]:
# @title 7. Package outputs (all scenes)
import shutil, os, glob
bundle = f'{WORK}/vf_nerf_outputs'
shutil.rmtree(bundle, ignore_errors=True)
os.makedirs(bundle, exist_ok=True)
for s in SCENES:
    run_dir = os.path.dirname(NERF_CONFIGS[s])
    shutil.copytree(run_dir, f'{bundle}/nerfacto_{s}',
                    ignore=shutil.ignore_patterns('*.tfevents*'))
    shutil.copytree(COND_DIR(s), f'{bundle}/conditional_nf_{s}')
    probe_pngs = glob.glob(f'{WORK}/explorer/{s}/probe_*.png')
    if probe_pngs:
        os.makedirs(f'{bundle}/probes_{s}', exist_ok=True)
        for png in probe_pngs:
            shutil.copy(png, f'{bundle}/probes_{s}')
        _dj = f'{WORK}/explorer/{s}/dino_consistency.json'
        if os.path.exists(_dj):
            shutil.copy(_dj, f'{bundle}/probes_{s}')
    for _m in glob.glob(f'{WORK}/eval/{s}_nerf_metrics.json'):
        shutil.copy(_m, bundle)
for mp4 in glob.glob(f'{RENDER_DIR}/*.mp4'):
    shutil.copy(mp4, bundle)
shutil.make_archive(bundle, 'zip', bundle)
print('bundle:', bundle + '.zip')
!du -sh {bundle}.zip {WORK}/renders {WORK}/checkpoints {WORK}/outputs
!find {bundle} -maxdepth 3 -type f | sort

## Getting your results

**Committed run (Save & Run All):** when the version finishes, open it and use the
**Output** tab — download `vf_nerf_outputs.zip` (per scene: nerfacto checkpoint +
config, conditional-NF `latest.pt`, probe montages, `<scene>_nerf_metrics.json`
with held-out PSNR/SSIM/LPIPS, and `probes_<scene>/dino_consistency.json` with the
source-vs-novel-view DINO cosine similarity, the source-point-to-sample distance, the
VF-NeRF mask coverage per view and the flow log-likelihood range over the pixels it
keeps) + the flythrough videos, or the individual files.

**Evaluation reports** each land in their own archive as soon as their cell finishes
(also listed in the Output tab):

| cell | archive | contents |
|---|---|---|
| 3c | `report_depth_probe.zip` | per scene: PNG (frame, camera moved to depth 0.3 / 3.0, depth map with contours) + JSON |
| 5b | `report_cond_nf_likelihood.zip` | `<scene>_cond_nf_ll.json` (raw / depth-filtered / shuffled-condition stats) + `_samples.npz` with every per-sample log-prob, depth, shuffled log-prob |
| 5c | `report_cond_nf_nearest_sample.zip` | `<scene>_cond_nf_nearest.json` (min / top-logp distance + angle stats, hit rates, depth-filtered) + `_samples.npz` per pixel |
| 6b | `report_explorer_probes.zip` | probe montages + `dino_consistency.json` per scene |
| 6c | `report_dino_recon.zip` | `<scene>_dino_recon.json` (same-patch vs random-patch cosine per frame) |

Reading 5b: use the **depth-filtered** mean/var (the raw ones are owned by a few
bad-depth rays -- the tail lines say how much), and compare the true mean against the
**shuffled** one: a small gap means the flow ignores the DINO feature. Reading 5c: `min`
is the best of 100 samples, `top` is the flow's own top-ranked sample -- the honest
single-render number. The standalone `notebooks/eval_cond_nf_likelihood.ipynb` and
`notebooks/eval_dino_reconstruction.ipynb` run the same two scripts against restored
checkpoints, for when only the evaluation is needed.

**Interactive run:** the file browser on the right shows `/kaggle/working/` — right-click
→ Download. Do a **Save Version** first so the outputs are also stored server-side.

If the GUI file browser / Output tab is unavailable, get a direct download link from a cell:

```python
from IPython.display import FileLink
import shutil; shutil.make_archive('/kaggle/working/vf_nerf_outputs', 'zip', '/kaggle/working/vf_nerf_outputs')
FileLink('vf_nerf_outputs.zip')   # click it (works while the session is alive)
```

## Loading an existing checkpoint into this notebook

Kaggle notebooks can only read outside files through an attached **Dataset** (or Secret).
There is no Drive mount. So:

1. **kaggle.com -> Create -> New Dataset.** Upload your checkpoint file(s). The scene
   is inferred from *directory structure*, matching exactly what cells 5/7 themselves
   produce -- not just any folder/filename that happens to mention the scene:
   - nerfacto: a path ending `.../<scene>/nerfacto/<timestamp>/config.yml` (the raw
     nerfstudio output tree), **or** `.../nerfacto_<scene>/config.yml` (the packaged
     bundle layout below).
   - conditional-NF: `.../conditional_nf/<scene>/latest.pt`, **or**
     `.../conditional_nf_<scene>/latest.pt` (packaged bundle layout).
   A prior run's `vf_nerf_outputs.zip` already uses the bundle layout
   (`nerfacto_<scene>/`, `conditional_nf_<scene>/`), so it can be re-attached as-is to
   resume every scene it contains at once -- just don't rename or flatten those folders
   before uploading:
   - a **nerfacto** backup: either a `*.tar.gz` of an `outputs/` tree (what the earlier
     Colab `tar_ckpts.py` produced), **or** just the loose `nerfacto/<timestamp>/` run
     dir (`config.yml` + `nerfstudio_models/*.ckpt` + `dataparser_transforms.json`).
     Either works -- Kaggle auto-extracts an uploaded archive, so a `.tar.gz` ends up
     as the loose tree anyway. **or**
   - a **conditional-NF** checkpoint: `latest.pt` or `cond_nf_step_*.pt` from
     `scripts/train_conditional_nf.py`.
   Set it Private if you like; give it any title.
2. In this notebook: right sidebar -> **Input -> Add Input** -> your dataset -> **Add**.
   It mounts read-only at `/kaggle/input/<dataset-slug>/`.
3. Re-run from **cell 2**. It auto-detects the files per scene:
   - nerfacto tarball or loose run dir found -> **cell 3 restores whichever scenes it
     can match by name**, repoints their baked-in `output_dir`, and skips their
     training;
   - `.pt` found -> **cell 5 copies it to that scene's `latest.pt` and skips its NF
     training** (the trainer has no resume flag, so a `.pt` is only *used*, not
     continued).

Every scene in `SCENES` (cell 0) still downloads + gets its downscaled images built in
**cell 2b** either way (needed for DINO features, the render camera path, and the
dataparser the config points at) -- restoring a checkpoint skips training, not data prep.

**External probe images** ride the same mechanism: put any images in a Dataset, attach
it, and reference them by filename in cell 6a's per-scene `COORDS` (e.g.
`["myphoto.jpg", 640, 480, "EXTERNAL"]`). Cell 6b resolves the name under
`/kaggle/input/**`,
conditions the flow on that pixel's DINO feature, and renders the sampled views through
that scene's frozen NeRF.

To get files *out* to Google Drive there's no built-in mount — download
`vf_nerf_outputs.zip` and upload it yourself, or add an `rclone`/Drive-API cell backed by
a Kaggle Secret.

## Interactive viewing — what works where

| tool | what it shows | Kaggle | needs |
|---|---|---|---|
| cell 4 video | spiral flythrough of the reconstructed NeRF | ✅ committed or interactive — **optional**, auto-skipped per scene when nerfacto is restored (set `FORCE_RENDER=True`) | nothing extra |
| `app/pick_points.py` + cells 6a/6b | click points on the images locally -> paste the `COORDS` block into 6a -> NF samples novel-view rays -> render top few per point (montage PNGs, 6b) | ✅ committed **or** interactive (picker runs on your machine) | both checkpoints, per scene |
| `ns-viewer` | free 3D navigation of the NeRF | ❌ not from Kaggle | browser must reach `ws://localhost:<port>` — i.e. local machine, or SSH port-forward from a rented GPU box (RunPod/Vast/Lambda) |

So: for a quick look at a scene's NeRF, force the cell 4 spiral render. To probe a
scene's conditional NF, run `python app/pick_points.py --scene <name>` on your own
machine -- it opens that scene's training images, you click the objects to probe, and it
prints a `COORDS` block to paste into **cell 6a** (under that scene's key); then run
**cell 6b**. (Kaggle's JupyterLab has no `ipympl` widget frontend, so there is no
in-notebook click capture; you can also hand-edit `COORDS` in 6a, which shows a pixel
grid per scene and draws your picks as circles.) Full free-fly `ns-viewer` needs a
machine you can point a browser at `localhost` on (local, or `ssh -L 7007:localhost:7007`
into a rented GPU box) — not Kaggle or Colab.